# Begin

In [1]:
# @launchit.collected
# @launchit.collected_temp_config
# @launchit.collected_optuna
# @launchit.collected_initrd
# @launchit.collected_build_docker_launch

In [2]:
# @launchit.collected_manual_run_docker_launch
# @launchit.collected_optuna_run_docker_launch

In [3]:
import os # @launchit.collect
import sys # @launchit.collect
import socket
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import math
import datetime
import json # @launchit.collect
import pprint # @launchit.collect
import re 
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import pickle # @launchit.collect
import IPython 
from enum import StrEnum, auto # @launchit.collect
import multiprocessing as mp
import queue

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np # @launchit.collect
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as VF
import torch.optim
import torch.multiprocessing as torch_mp
from torch.distributions import Categorical
from torch.nn.attention import SDPBackend, sdpa_kernel

import gymnasium as gym
import ale_py
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
import av

import optuna # @launchit.collect
from optuna.storages import JournalStorage # @launchit.collect
from optuna.storages.journal import JournalFileBackend # @launchit.collect
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
build_project_root_path = '${BUILD_PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
sys.path.append(os.path.join(build_project_root_path, 'lib')) # @launchit.collect
from cleanrl.cleanrl_utils.atari_wrappers import (  # isort:skip
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)
import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter, RecursiveMovingAverageFilter
from logging_utils import *
from artifact_registry import * # @launchit.collect
import torch_utils
import launchit 
from hp_utils import * # @launchit.collect
from metrics_collector import RmqSummaryWriter, S3SummaryWriter
from autoincrement import Autoincrement
import command_listener
import ob_preprocessor

# Init

In [4]:
# @launchit.collect
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    DOCKER_LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

In [5]:
# @launchit.collect_temp_config
# @launchit.disable

# Construct temporary CONFIG object for build and bootstrap purposes
if '${LAUNCHIT_FNAME}' != '$' + '{LAUNCHIT_FNAME}':
    notebook_fname = '${LAUNCHIT_FNAME}'
    notebook_basename = os.path.basename(notebook_fname)
    notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    subproject_name = os.path.basename(os.path.dirname(notebook_fname))
    
    if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
        # This (bootstrap) config is for the very initial phase of docker launch when one needs to load hyperparameters.
        # This config will be recreated soon to full-fledged Config
        CONFIG = namedtuple('BootstrapConfig', 'initrd_path, exec_mode')(
            initrd_path=os.path.join(project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK,
        )
        assert Logging._instance is None, 'Must create Logging instance with use_raw_stdout=True, but Logging instance is already created'
        # use_raw_stdout is necessary because sys.stdout is hijacked by papermill and this will lead to problems
        # with string duplications written to stdout by child process (worker)
        # see https://share.google/aimode/ffIcoqyaJdgMsEcoW
        Logging.get(use_raw_stdout=True, log_fname=os.path.join(project_root_path, 'run', subproject_name, notebook_name + '.log'))
    else:
        # Config to build docker launch
        CONFIG = namedtuple('BuildConfig', 
                            'initrd_path, relative_initrd_path, relative_run_path, relative_metrics_suite_fname, ' + 
                            'model_group_uri, self_fname, relative_self_fname, self_name, subproject_name, ' + 
                            'docker_registry, exec_mode, ' +
                            'artifact_registry_cache_fname, artifact_registry_cache, artifact_registry')(
            initrd_path=os.path.join(build_project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            relative_initrd_path=os.path.join('run', subproject_name, 'initrd-' + notebook_name), # relative to build project root
            relative_run_path=os.path.join('run', subproject_name),
            relative_metrics_suite_fname=os.path.join('run', subproject_name, notebook_name + '.metrics_suite.json'),
            model_group_uri='${MODEL_GROUP_URI}',
            self_fname=notebook_fname,
            relative_self_fname=lu.when('/run/' in notebook_fname, 'run/', '') + os.path.join(subproject_name, notebook_basename),  # relative to build project root
            self_name=notebook_name,
            subproject_name=subproject_name,
            docker_registry='cr.selcloud.ru/neurolab',
            exec_mode=ExecMode.LAUNCH_NOTEBOOK,
            artifact_registry_cache_fname=None, 
            artifact_registry_cache={}, 
            artifact_registry=None,
        )
        CONFIG = CONFIG._replace(artifact_registry=ArtifactRegistry(CONFIG.model_group_uri, cache=CONFIG.artifact_registry_cache))
        CONFIG = CONFIG._replace(artifact_registry_cache_fname=os.path.join(CONFIG.initrd_path, 'artifact_registry_cache.pkl'))
        os.makedirs(CONFIG.initrd_path, exist_ok=True)

    Logging.get()(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n')
# @launchit.stop

In [6]:
def create_config():
    config = namedtuple('Config', 
                        'host_name, ' +
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, initrd_path, ' + 
                        'self_fname, self_name, metrics_suite_fname, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, docker_registry, exec_mode, is_interactive')(
        host_name=socket.gethostname(),
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        initrd_path=None,
        self_fname=None,
        self_name=None,
        metrics_suite_fname=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cpu',
        docker_registry='cr.selcloud.ru/neurolab',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )

    env_cuda_device = os.environ.get('CUDA_DEVICE')
    
    if torch.cuda.is_available():
        default_cuda_device = 'cuda'
        config = config._replace(cuda_device=lu.coalesce(env_cuda_device, default_cuda_device))
    else:
        assert env_cuda_device is None, f'CUDA device "{env_cuda_device}" is requested but CUDA is NOT available!'
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf).get('jupyter_session')

            if notebook_fname is None:
                notebook_fname = os.path.join(config.subproject_path, os.path.basename('${LAUNCHIT_FNAME}'))
                assert os.path.exists(notebook_fname)
            
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None

            if is_launch:
                if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
                    config = config._replace(exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK)
                else:
                    config = config._replace(exec_mode=ExecMode.LAUNCH_NOTEBOOK)
            else:
                assert config.exec_mode == ExecMode.MASTER_NOTEBOOK
    
    config = config._replace(is_interactive=config.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(initrd_path=os.path.join(project_root_path, 'run', config.subproject_name, 'initrd-' + config.self_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    config = config._replace(metrics_suite_fname=os.path.join(config.run_path, config.self_name + '.metrics_suite.json'))
    return config

In [7]:
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', CONFIG.exec_mode == ExecMode.LAUNCH_MODULE)
LOG.enable('stdout', CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
LOG.enable('verbose_stdout', CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
LOG.enable('file', CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
LOG(f'{os.environ=}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)
os.makedirs(CONFIG.initrd_path, exist_ok=True)

CONFIG=
{'host_name': 'baki',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'initrd_path': '/home/misha/dev/mine/neurolab/run/18_rl/initrd-18n_ppo_tr_frostbite_07',
 'self_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07.ipynb',
 'self_name': '18n_ppo_tr_frostbite_07',
 'metrics_suite_fname': '/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07.metrics_suite.json',
 'subproject_name': '18_rl',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'docker_registry': 'cr.selcloud.ru/neurolab',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [8]:
# @launchit.disable
# @launchit.collect
LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: str = None
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    @dataclass(slots=True)
    class General:
        comment: str = None
        random_seed: int = None
        is_torch_deterministic: bool = True
        is_torch_compile: bool = False
        is_torch_amp: bool = False

    @dataclass(slots=True)
    class Env:
        count: int = 1 # number of parallel game environments
        is_episodic_life: bool = None
        actions_count: int = None # None - consider all actions, otherwise take only first :actions_count

    @dataclass(slots=True)
    class VisionHead:
        parent: dict = None # params: model, weights
        is_trainable: bool = None
    
    @dataclass(slots=True)
    class Encoder:
        parent: dict = None # params: model, weights
        is_trainable: bool = None
    
    @dataclass(slots=True)
    class Agent:
        parent: str = None 
        sequence_length: int = 4 # observations chain max length
        action_plan_length: int = 10 # number of actions agent must think upfront about
        d_model: int = 256 # dimension of the transformer
        transformer: dict = None # dict with params: layers_count, heads_count, attention_backend
        is_trainable: bool = None

    @dataclass(slots=True)
    class Test:
        env_rams: list = None 
        env_ram_patches: list = None 
        break_on_level_passed: bool = None 
        
    @dataclass(slots=True)
    class PPO:
        global_steps_count: int = 1_000_000 # total number of steps 
        rollout_steps_count: int = 512 # how many steps to run in a single policy rolllout
        rollout_env_rams: list = None # list of RAM to randomly choose on reset during rollout (a-la "Load Game" feature)
        rollout_env_ram_patches: list = None # patches to randomly choose and apply to RAM to get more diverse exploration space
        rollout_env_stories: list = dataclasses.field(default_factory=lambda: ['*;*']) # story = RAM + RAM patch. '*' = any RAM/RAM patch
        
        epochs_count: int = 3 
        batch_size: int = 128
        learn_rate: str = None
        optimizer: str = 'AdamW'
        
        vf_coef: float = 0.5 # coefficient of the value function within loss function
        ent_coef: str = None # coefficient of the entropy member  within loss function
        consistency_coef: float = 0.0 # coefficient of the loss for action plan consistency between adjacent steps
        prediction_coef: float = 0.0 # coefficient of the loss for observation emb. prediction, if set to 0.0 the prediction loss is not used
        
        gamma: float = 0.995 # return discount factor gamma
        gae_lambda: float = 0.95 # lambda for the general advantage estimation
        clip_coef: float = 0.1 # the surrogate clipping coefficient
        clip_vloss: bool = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
        max_grad_norm: float = 0.25 # the maximum norm for the gradient clipping
        target_kl: float = None # e target KL divergence threshold
        norm_adv: bool = False # Toggles advantages normalization

    general: General = dataclasses.field(default_factory=General)
    env: Env = dataclasses.field(default_factory=Env)
    vision_head: VisionHead = dataclasses.field(default_factory=VisionHead)
    encoder: Encoder = dataclasses.field(default_factory=Encoder)
    agent: Agent = dataclasses.field(default_factory=Agent)
    test: Test = dataclasses.field(default_factory=Test)
    ppo: PPO = dataclasses.field(default_factory=PPO)

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        hp.general = Hyperparameters.General(**hp.general)
        hp.env = Hyperparameters.Env(**hp.env)
        hp.vision_head = Hyperparameters.VisionHead(**hp.vision_head)
        hp.encoder = Hyperparameters.Encoder(**hp.encoder)
        hp.agent = Hyperparameters.Agent(**hp.agent)
        hp.test = Hyperparameters.Test(**hp.test)
        hp.ppo = Hyperparameters.PPO(**hp.ppo)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()

# Runtime

## Runtime

In [9]:
@dataclass(slots=True)
class Runtime:
    command_listener: object = None
    should_save_model: bool = True
    mp_ctx: object = None
    optuna_trial: dict = None
    artifact_registry_cache: dict = None
    artifact_registry: object = None
    summary_writer: object = None
    env_observation_space_shape: object = None
    env_action_space_shape: object = None
    vision_head: object = None
    encoder: object = None
    agent: object = None
    test_manager: object = None

## Configure

In [10]:
# @launchit.disable
# @launchit.collect
HP.launch_goal = 'train'
import random
HP.general.comment = None
HP.general.random_seed = random.randint(0, 100)
HP.general.is_torch_deterministic = True
HP.general.is_torch_compile = False
HP.general.is_torch_amp = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': 'train',
 'launch_id': 0,
 'general': {'comment': None,
             'random_seed': 27,
             'is_torch_deterministic': True,
             'is_torch_compile': False,
             'is_torch_amp': True},
 'env': {'count': 1, 'is_episodic_life': None, 'actions_count': None},
 'vision_head': {'parent': None, 'is_trainable': None},
 'encoder': {'parent': None, 'is_trainable': None},
 'agent': {'parent': None,
           'sequence_length': 4,
           'action_plan_length': 10,
           'd_model': 256,
           'transformer': None,
           'is_trainable': None},
 'test': {'env_rams': None,
          'env_ram_patches': None,
          'break_on_level_passed': None},
 'ppo': {'global_steps_count': 1000000,
         'rollout_steps_count': 512,
         'rollout_env_rams': None,
         'rollout_env_ram_patches': None,
         'rollout_env_stories': ['*;*'],
         'epochs_count': 3,
         'batch_size': 128,
         'learn_rate': None,
         'optimizer':

## Create

In [11]:
assert type(CONFIG).__name__ == 'Config', 'Runtime should be created for full-fledged Config instance only'
LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)

RT = Runtime()
RT.command_listener = command_listener.CommandListener()

if 'COMMAND_LISTENER' in os.environ:
    RT.command_listener.start()
    LOG(f'CommandListener started on port={RT.command_listener.DEFAULT_PORT}')

RT.mp_ctx = torch_mp.get_context('spawn') # Spawn is needed for CUDA, fork doesn't work within PyTorch
    
if HP.general.random_seed is not None:
    random.seed(HP.general.random_seed)
    torch.manual_seed(HP.general.random_seed)
    RNG = np.random.default_rng(HP.general.random_seed)    
    LOG(f'Random seed={HP.general.random_seed}')

if HP.general.is_torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.general.is_torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

torch._functorch.config.donated_buffer = False
LOG(f'{torch._functorch.config.donated_buffer=}')

lc = HP.launch_component()

artifact_registry_type = os.environ.get('ARTIFACT_REGISTRY', '').upper()

if not artifact_registry_type:
    artifact_registry_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'NEXUS')

match artifact_registry_type:
    case 'NEXUS':
        kwargs = {}

        if os.environ.get('NEXUS_URL', None) is not None:
            kwargs['nexus_url'] = os.environ['NEXUS_URL']

        if os.environ.get('DOWNLOAD_NEXUS_URL', None) is not None:
            kwargs['download_nexus_url'] = os.environ['DOWNLOAD_NEXUS_URL']
        
        RT.artifact_registry = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri, **kwargs)
        LOG(f'ArtifactRegistry created')
    case 'S3':
        kwargs = {}
        artifact_registry_cache_fname = os.path.join(CONFIG.initrd_path, 'artifact_registry_cache.pkl')
        
        if os.path.exists(artifact_registry_cache_fname):
            with open(artifact_registry_cache_fname, 'rb') as f:
                RT.artifact_registry_cache = pickle.load(f)
                assert isinstance(RT.artifact_registry_cache, dict)
                LOG(f'Artifact registry cache loaded from "{artifact_registry_cache_fname}"')
        
            kwargs['cache'] = RT.artifact_registry_cache
        else:
            LOG(f'Artifact registry cache "{artifact_registry_cache_fname}" is absent, nothing to load')
        
        RT.artifact_registry = S3ArtifactRegistry(maven_group_id=CONFIG.model_group_uri, **kwargs)
        LOG(f'S3ArtifactRegistry created')
    case _:
        assert False, f'Unsupported {artifact_registry_type=}'

if lc.version != 0:
    assert CONFIG.exec_mode != ExecMode.MASTER_NOTEBOOK, 'With MASTER_NOTEBOOK exec_mode one should not overwrite any of the launches (experiments)'
    RT.artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    meta = dict(
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)
else:
    assert CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK, 'MASTER_NOTEBOOK exec_mode is for working on 0 (dummy) version only'

RT.optuna_trial = None
optuna_trial_fname = os.path.join(CONFIG.initrd_path, 'optuna_trial.json')

if os.path.exists(optuna_trial_fname):
    with open(os.path.join(optuna_trial_fname), 'rt') as f:
        RT.optuna_trial = json.load(f)
        assert 'trial_number' in RT.optuna_trial, RT.optuna_trial
        assert 'study_serial' in RT.optuna_trial, RT.optuna_trial
        assert 'study_name' in RT.optuna_trial, RT.optuna_trial

    LOG(f'Optuna trial loaded from "{optuna_trial_fname}": {RT.optuna_trial}')

summary_log_dir = lc.name

if RT.optuna_trial is not None:
    summary_log_dir = os.path.join(summary_log_dir, f'opt_{RT.optuna_trial['study_serial']}')
    
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')

summary_writer_type = os.environ.get('SUMMARY_WRITER', '').upper()

if not summary_writer_type:
    summary_writer_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'RMQ')

match summary_writer_type:
    case 'RMQ':
        kwargs = {}

        if os.environ.get('RMQ_CONNECTION_URL', None) is not None:
            kwargs['rmq_connection_url'] = os.environ['RMQ_CONNECTION_URL']

        RT.summary_writer = RmqSummaryWriter(log_dir=summary_log_dir, **kwargs)
        LOG(f'RmqSummaryWriter created')
    case 'S3':
        RT.summary_writer = S3SummaryWriter(log_dir=summary_log_dir)
        LOG(f'S3SummaryWriter created')
    case _:
        assert False, f'Unsupported {summary_writer_type=}'

RT.summary_writer.add_text('hyperparameters', pprint.pformat(HP._asdict(), sort_dicts=False), 0)
RT.summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 0)
RT.summary_writer.flush()

Random seed=27
torch.backends.cudnn.deterministic=True
torch._functorch.config.donated_buffer=False
ArtifactRegistry created
Tensorboard run=18n_ppo_tr_frostbite_07/0
RmqSummaryWriter created


# Environment

In [17]:
# @launchit.collect_export
import gymnasium as gym
import ale_py

## MyUberWrapper

In [18]:
# @launchit.collect_export
class MyUberWrapper(gym.vector.VectorWrapper):
    def __init__(self, env, idle_penalty, life_lost_penalty, frame_skip):
        super().__init__(env)
        assert isinstance(env, ale_py.AtariVectorEnv)
        self.frame_numbers = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.episode_lengths = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.episode_returns = np.zeros(env.unwrapped.num_envs)
        self.lives = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.was_life_lost = np.zeros(env.unwrapped.num_envs, dtype=np.bool)
        self.life_lengths = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.life_returns = np.zeros(env.unwrapped.num_envs)
        assert idle_penalty <= 0
        assert life_lost_penalty <= 0
        self.idle_penalty = idle_penalty
        self.life_lost_penalty = life_lost_penalty
        self.frame_skip = frame_skip # forward data, used within video capture to compute fps
        self.levels_passed = np.zeros(env.unwrapped.num_envs, dtype=np.int32)

    def step(self, actions):
        self.life_lengths[self.was_life_lost] = 0
        self.life_returns[self.was_life_lost] = 0
        
        obs, rewards, terminations, truncations, infos = self.env.step(actions)

        frame_numbers = infos['frame_number'] # grows indefinitely starting from ROM load or from state load
        episode_lengths = infos['episode_frame_number'] # resets on every new episode (game)
        lives = infos['lives']

        addon_lenghts = frame_numbers - self.frame_numbers
        self.frame_numbers[:] = frame_numbers
        assert np.all(addon_lenghts >= 0), addon_lenghts
        
        infos['is_episode_over'] = np.logical_or(terminations, truncations)
        self.episode_lengths[:] = episode_lengths
        self.episode_returns += rewards

        # Logic of screensaver (https://share.google/aimode/27PlUFMZg8WEpCBWR) moved to ale_py
        # is_screensaver_activated = (lives == 4 and self.lives == 1)
        # infos['is_life_lost'] = (lives < self.lives) | (is_screensaver_activated) | infos['is_episode_over']
        infos['is_life_lost'] = (lives < self.lives) | infos['is_episode_over']
        self.was_life_lost[:] = infos['is_life_lost']
        self.lives[:] = lives
        
        self.life_lengths += addon_lenghts
        self.life_returns += rewards

        infos['episode_lengths'] = self.episode_lengths.copy()
        infos['episode_returns'] = self.episode_returns.copy()
        infos['life_lengths'] = self.life_lengths.copy()
        infos['life_returns'] = self.life_returns.copy()

        clipped_rewards = np.sign(rewards)

        if self.idle_penalty < 0:
            clipped_rewards = np.where(clipped_rewards == 0, self.idle_penalty, clipped_rewards)

        if self.life_lost_penalty < 0:
            clipped_rewards[infos['is_life_lost']] = self.life_lost_penalty

        infos['is_level_passed'] = infos['levels_passed'] > self.levels_passed
        self.levels_passed = infos['levels_passed']

        return self._fix_obs(obs), clipped_rewards, terminations, truncations, infos

    def reset(self, seed=None, options=None):
        obs, infos = self.env.reset(seed=seed, options=options)
        reset_mask = lu.coalesce(options, {}).get('reset_mask')

        if reset_mask is None:
            reset_mask = np.full(len(self.episode_lengths), True, dtype=np.bool)

        self.frame_numbers[reset_mask] = infos['frame_number'][reset_mask]
        self.episode_lengths[reset_mask] = 0
        self.episode_returns[reset_mask] = 0
        self.life_lengths[reset_mask] = 0
        self.life_returns[reset_mask] = 0
        self.was_life_lost[reset_mask] = False
        self.levels_passed[reset_mask] = 0

        return self._fix_obs(obs), infos

    def _fix_obs(self, obs):
        # Get rid of dummy dimension cast by degenerate stack frames=1
        assert obs.ndim == 5, obs.shape # [num_envs, stack_size, height, width, 3]
        return obs.reshape(obs.shape[0], *obs.shape[2:])        

    def __repr__(self):
        return f'<{self.__class__.__name__}(idle_penalty={self.idle_penalty}, life_lost_penalty={self.life_lost_penalty}), {self.env}>'

## EnvRamPatcher

In [19]:
# @launchit.collect_export
class EnvRamPatcher:
    def __call__(self, patch):
        assert patch is None or isinstance(patch, list)
        result = {}

        if patch is None:
            return result

        for p in patch:
            func = getattr(self, p)
            func(result)

        return result
    
    def no_score(self, ram):
        ram[73] = 0 
        ram[74] = 0
    
    def last_life(self, ram):
        ram[76] = 0

    def three_lives(self, ram):
        ram[76] = 3

    def eight_lives(self, ram):
        ram[76] = 8

    def nine_lives(self, ram):
        ram[76] = 9

    def full_igloo(self, ram):
        ram[77] = 15

    def half_igloo(self, ram):
        ram[77] = 7

    def one_remaining_igloo(self, ram):
        ram[77] = 14

    def three_remaining_igloo(self, ram):
        ram[77] = 12

    def no_igloo(self, ram):
        ram[77] = 255

    def temperature_10(self, ram):
        ram[101] = 10

    def temperature_20(self, ram):
        ram[101] = 32

    def temperature_45(self, ram):
        ram[101] = 70

    def bailey_right_at_the_igloo_door(self, ram):
        ram[102] = 124

    def bailey_very_near_igloo_door(self, ram):
        ram[102] = RNG.integers(108, 137, endpoint=True).item()

    def bailey_to_the_left_of_igloo(self, ram):
        ram[102] = RNG.integers(115, 125, endpoint=True).item()
    
    def bailey_to_the_right_of_igloo(self, ram):
        ram[102] = RNG.integers(130, 150, endpoint=True).item()

    def bailey_near_center(self, ram):
        ram[102] = RNG.integers(45, 95, endpoint=True).item()

    def bailey_random_spawn(self, ram):
        ram[102] = RNG.integers(16, 150, endpoint=True).item()

    def bailey_safe_random_spawn(self, ram):
        ram[102] = RNG.integers(16, 103, endpoint=True).item()

    def bailey_left_spawn(self, ram):
        ram[102] = 16

    def bailey_right_spawn(self, ram):
        ram[102] = 150

    def bear_to_the_left_of_igloo(self, ram):
        ram[104] = RNG.integers(25, 75, endpoint=True).item()

    def bear_chases_bailey_to_the_left_of_igloo(self, ram):
        ram[104] = RNG.integers(25, 75, endpoint=True).item()
        ram[102] = ram[104] + 25

    def bear_chases_bailey_to_the_left_of_igloo_2(self, ram):
        ram[104] = RNG.integers(25, 75, endpoint=True).item()
        ram[102] = 110

    def bear_chases_bailey_to_the_right_of_igloo(self, ram):
        ram[104] = RNG.integers(145, 150, endpoint=True).item()
        ram[102] = ram[104] - 15

## create_envs

In [20]:
# @launchit.collect_export
def create_envs(envs_count, thread_pool_size=None, thread_affinity_offset=None, is_strict_disabled_autoreset=True):
    env_id = 'frostbite'   
    frame_skip = 4
    
    envs = ale_py.AtariVectorEnv(
        game=env_id, 
        stack_num=1, # frame stacking is not needed because we use Transformer
        maxpool=False, # Use non-pooled observations!!!
        frameskip=frame_skip, # frameskip=4 corresponds to MaxAndSkipEnv(env, skip=4)
        repeat_action_probability=0, # corresponds NoFrameskip-v4
        use_fire_reset=True, # FireResetEnv(env)
        noop_max=30, # NoopResetEnv(env, noop_max=30)
        
        episodic_life=False, # will manage episodic life our selves
        life_loss_info=False, # strange parameter. When set to True then program will segfault
        reward_clipping=False, # will clip reward in wrapper in order to keep original rewards for game stats accounting

        # Get raw observation since rescaling and grayscaling is done on GPU
        grayscale=False,
        img_height=210,
        img_width=160,
        
        num_envs=envs_count, 
        num_threads=lu.coalesce(thread_pool_size, 0), # 0 means num of threads = num of envs
        thread_affinity_offset=lu.coalesce(thread_affinity_offset, -1), # -1 means no affinity

        autoreset_mode=gym.vector.vector_env.AutoresetMode.DISABLED,
        is_strict_disabled_autoreset=is_strict_disabled_autoreset,
    )

    return MyUberWrapper(
        envs, 
        idle_penalty=0, 
        life_lost_penalty=0,
        frame_skip=frame_skip, # used in video capturing to compute fps
    )

## Configure 

In [21]:
# @launchit.disable
# @launchit.collect
HP.env.count = 32 
HP.env.is_episodic_life = True
HP.env.actions_count = 6
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': 'train',
 'launch_id': 0,
 'general': {'comment': None,
             'random_seed': 48,
             'is_torch_deterministic': True,
             'is_torch_compile': False,
             'is_torch_amp': True},
 'env': {'count': 32, 'is_episodic_life': True, 'actions_count': 6},
 'vision_head': {'parent': None, 'is_trainable': None},
 'encoder': {'parent': None, 'is_trainable': None},
 'agent': {'parent': None,
           'sequence_length': 4,
           'action_plan_length': 10,
           'd_model': 256,
           'transformer': None,
           'is_trainable': None},
 'test': {'env_rams': None,
          'env_ram_patches': None,
          'break_on_level_passed': None},
 'ppo': {'global_steps_count': 1000000,
         'rollout_steps_count': 512,
         'rollout_env_rams': None,
         'rollout_env_ram_patches': None,
         'rollout_env_stories': ['*;*'],
         'epochs_count': 3,
         'batch_size': 128,
         'learn_rate': None,
         'optimizer': '

## Create

In [22]:
envs = create_envs(1)
LOG(f'Test envs created: {envs}')
assert isinstance(envs.single_action_space, gym.spaces.Discrete), f'Only discrete action spaces are supported, but got {type(env.single_action_space)}'
assert envs.single_observation_space.shape == (1, 210, 160, 3)
RT.env_observation_space_shape = envs.single_observation_space.shape
RT.env_action_space_shape = (lu.coalesce(HP.env.actions_count, envs.single_action_space.n.item()),)
assert len(RT.env_action_space_shape) == 1

LOG(f'{envs.metadata=}')
LOG(f'{RT.env_observation_space_shape=}')
LOG(f'{RT.env_action_space_shape=}')

del envs

Test envs created: <MyUberWrapper(idle_penalty=0, life_lost_penalty=0), AtariVectorEnv(num_envs=1)>
envs.metadata={'autoreset_mode': <AutoresetMode.DISABLED: 'Disabled'>}
RT.env_observation_space_shape=(1, 210, 160, 3)
RT.env_action_space_shape=(6,)


# VisionHead

## Configure

In [23]:
# @launchit.disable
# @launchit.collect
HP.vision_head.parent = dict(model='18d_world_model_09:40', weights='18d_world_model_09:40')
HP.vision_head.is_trainable = False
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': 'train',
 'launch_id': 0,
 'general': {'comment': None,
             'random_seed': 48,
             'is_torch_deterministic': True,
             'is_torch_compile': False,
             'is_torch_amp': True},
 'env': {'count': 32, 'is_episodic_life': True, 'actions_count': 6},
 'vision_head': {'parent': {'model': '18d_world_model_09:40',
                            'weights': '18d_world_model_09:40'},
                 'is_trainable': False},
 'encoder': {'parent': None, 'is_trainable': None},
 'agent': {'parent': None,
           'sequence_length': 4,
           'action_plan_length': 10,
           'd_model': 256,
           'transformer': None,
           'is_trainable': None},
 'test': {'env_rams': None,
          'env_ram_patches': None,
          'break_on_level_passed': None},
 'ppo': {'global_steps_count': 1000000,
         'rollout_steps_count': 512,
         'rollout_env_rams': None,
         'rollout_env_ram_patches': None,
         'rollout_env_stories': ['*;*

## Initrd

In [24]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig':
    coords = hp_parse_artifact_source(HP.vision_head.parent['model'])
    CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='ipynb', maven_group_id=coords.group_id)
    CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='json', asset_classifier='vision_head_params', maven_group_id=coords.group_id)

    coords = hp_parse_artifact_source(HP.vision_head.parent['weights'])
    CONFIG.artifact_registry.get_asset_content(
        coords.model_name, 
        coords.model_version, 
        asset_ext='pt', 
        asset_classifier=lu.coalesce(coords.asset_classifier, 'vision_head'), 
        maven_group_id=coords.group_id,
    )
# @launchit.stop

## Create

In [25]:
coords = hp_parse_artifact_source(HP.vision_head.parent['model'])

notebook_data = RT.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='ipynb', maven_group_id=coords.group_id)
source_code = launchit.extract_source_code(io.BytesIO(notebook_data), collect_inds=[None, 'export'])
source_code = 'import ob_preprocessor\n' + source_code
module = lu.make_module(f'{HP.vision_head.parent}_notebook', source_code)
module.CONFIG = CONFIG
module.RNG = RNG
module.LOG = LOG

params = RT.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier='vision_head_params', asset_ext='json', maven_group_id=coords.group_id)
params = module.VisionHead.Params(**json.loads(params.decode('utf-8')))
RT.vision_head = module.VisionHead(params).to(CONFIG.cuda_device)
LOG(f'VisionHead created from "{HP.vision_head.parent['model']}"')

coords = hp_parse_artifact_source(HP.vision_head.parent['weights'])
pt_data = RT.artifact_registry.get_asset_content(
    coords.model_name, 
    coords.model_version, 
    asset_ext='pt', 
    asset_classifier=lu.coalesce(coords.asset_classifier, 'vision_head'), 
    maven_group_id=coords.group_id,
)

with io.BytesIO(pt_data) as b:
    kwargs = {}
    
    if not CONFIG.is_cuda:
        kwargs['map_location'] = torch.device('cpu')

    state_dict = torch.load(b, **kwargs)
    RT.vision_head.load_state_dict(state_dict)

LOG(f'VisionHead state loaded from "{HP.vision_head.parent['weights']}"')

if HP.general.is_torch_compile:
    RT.vision_head = torch.compile(RT.vision_head, fullgraph=True)
    LOG(f'VisionHead compiled')

RT.vision_head.is_trainable = HP.vision_head.is_trainable
assert next(iter(RT.vision_head.parameters())).requires_grad == HP.vision_head.is_trainable
LOG(f'{RT.vision_head.is_trainable=}')

VisionHead created from "18d_world_model_09:40"
VisionHead state loaded from "18d_world_model_09:40"
RT.vision_head.is_trainable=False


# Encoder

## Configure

In [26]:
# @launchit.disable
# @launchit.collect
HP.encoder.parent = dict(model='18d_world_model_09:40', weights='18d_world_model_09:40')
HP.encoder.is_trainable = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': 'train',
 'launch_id': 0,
 'general': {'comment': None,
             'random_seed': 48,
             'is_torch_deterministic': True,
             'is_torch_compile': False,
             'is_torch_amp': True},
 'env': {'count': 32, 'is_episodic_life': True, 'actions_count': 6},
 'vision_head': {'parent': {'model': '18d_world_model_09:40',
                            'weights': '18d_world_model_09:40'},
                 'is_trainable': False},
 'encoder': {'parent': {'model': '18d_world_model_09:40',
                        'weights': '18d_world_model_09:40'},
             'is_trainable': True},
 'agent': {'parent': None,
           'sequence_length': 4,
           'action_plan_length': 10,
           'd_model': 256,
           'transformer': None,
           'is_trainable': None},
 'test': {'env_rams': None,
          'env_ram_patches': None,
          'break_on_level_passed': None},
 'ppo': {'global_steps_count': 1000000,
         'rollout_steps_count': 512,
         'r

## Initrd

In [27]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig':
    coords = hp_parse_artifact_source(HP.encoder.parent['model'])
    CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='ipynb', maven_group_id=coords.group_id)
    CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='json', asset_classifier='encoder_params', maven_group_id=coords.group_id)

    coords = hp_parse_artifact_source(HP.encoder.parent['weights'])
    CONFIG.artifact_registry.get_asset_content(
        coords.model_name, 
        coords.model_version, 
        asset_ext='pt', 
        asset_classifier=lu.coalesce(coords.asset_classifier, 'encoder'), 
        maven_group_id=coords.group_id,
    )
# @launchit.stop

## Create

In [28]:
coords = hp_parse_artifact_source(HP.encoder.parent['model'])

notebook_data = RT.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='ipynb', maven_group_id=coords.group_id)
source_code = launchit.extract_source_code(io.BytesIO(notebook_data), collect_inds=[None, 'export'])
source_code = 'import torch_utils\n' + source_code
module = lu.make_module(f'{HP.encoder.parent}_notebook', source_code)
module.CONFIG = CONFIG
module.RNG = RNG
module.LOG = LOG

params = RT.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier='encoder_params', asset_ext='json', maven_group_id=coords.group_id)
params = module.Encoder.Params(**json.loads(params.decode('utf-8')))
RT.encoder = module.Encoder(params).to(CONFIG.cuda_device)
LOG(f'Encoder created from "{HP.encoder.parent['model']}"')

coords = hp_parse_artifact_source(HP.encoder.parent['weights'])
pt_data = RT.artifact_registry.get_asset_content(
    coords.model_name, 
    coords.model_version, 
    asset_ext='pt', 
    asset_classifier=lu.coalesce(coords.asset_classifier, 'encoder'), 
    maven_group_id=coords.group_id,
)

with io.BytesIO(pt_data) as b:
    kwargs = {}
    
    if not CONFIG.is_cuda:
        kwargs['map_location'] = torch.device('cpu')

    state_dict = torch.load(b, **kwargs)
    RT.encoder.load_state_dict(state_dict)

LOG(f'Encoder state loaded from "{HP.encoder.parent['weights']}"')

if HP.general.is_torch_compile:
    RT.encoder = torch.compile(RT.encoder, fullgraph=True)
    LOG(f'Encoder compiled')

RT.encoder.is_trainable = HP.encoder.is_trainable
assert next(iter(RT.encoder.parameters())).requires_grad == HP.encoder.is_trainable
LOG(f'{RT.encoder.is_trainable=}')

Encoder created from "18d_world_model_09:40"
Encoder state loaded from "18d_world_model_09:40"
RT.encoder.is_trainable=True


# Agent

In [29]:
# @launchit.collect_export
from collections import namedtuple
from dataclasses import dataclass
import math

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.functional as VF
from torch.distributions import Categorical
from torch.nn.attention import SDPBackend, sdpa_kernel

## PredictorHead

In [30]:
# @launchit.collect_export
class PredictorHead(nn.Module):
    @dataclass(slots=True)
    class Params:
        actions_count: int = 6
        d_model: int = 256
        d_inner: int = 64
        
    def __init__(self, params):
        super().__init__()
        self.params = params
        self.action_embedding = nn.Embedding(self.params.actions_count, self.params.d_inner) 
        self.network = nn.Sequential(
            nn.Linear(self.params.d_model + self.params.d_inner, self.params.d_model),
            nn.ReLU(),
            nn.Linear(self.params.d_model, self.params.d_model),
        )

    def forward(self, latents, actions):
        # latents: (..., d_model)
        # actions: (...)
        latents_shape = latents.shape
        latents = latents.view(-1, self.params.d_model)
        actions = actions.view(-1)
        assert len(latents) == len(actions)
        action_latents = self.action_embedding(actions) # (uberbatch, d_inner)
        all_latents = torch.cat([latents, action_latents], dim=-1) # (uberbatch, d_model + d_inner]
        predicted_latents = self.network(all_latents) # (uberbatch, d_model)
        return predicted_latents.view(latents_shape)

## Agent

In [31]:
# @launchit.collect_export
class Agent(torch_utils.TrainableModule):
    @dataclass(slots=True)
    class Params:
        sequence_length: int = 4
        actions_count: int = 6
        action_plan_length: int = 10 # how many actions to think of ahead
        d_model: int = 256
        transformer: dict = None

    def __init__(self, params):
        super().__init__()
        self.params = params
        assert self.params.sequence_length > 0
        assert self.params.actions_count > 0
        assert self.params.action_plan_length > 0

        self.temporal_encoding = nn.Parameter(torch.randn(1, self.params.sequence_length, self.params.d_model)) # seq number (0,1,2,3...) -> embedding
        self.action_plan_encoding = nn.Parameter(torch.randn(1, self.params.action_plan_length, self.params.d_model))
        self.internal_sequence_length = (
            self.params.sequence_length + # observations
            self.action_plan_encoding.shape[1] 
        )

        attention_mask = self.generate_attention_mask()
        self.register_buffer('attention_mask', attention_mask)

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=self.params.d_model, 
            dim_feedforward=self.params.d_model * 4, # kms@ default is 2048, 384 * 4 = 1536
            nhead=self.params.transformer['heads_count'], 
            batch_first=True,
            norm_first=True,  # preferred for RL tasks
            dropout=0.0,      # Dropout destroys RL performance; keep 0.0
        )
        attention_backend_names = lu.when(
            isinstance(self.params.transformer['attention_backend'], str), 
            [self.params.transformer['attention_backend']], 
            self.params.transformer['attention_backend'],
        )
        self.attention_backend = list(map(lambda x: getattr(SDPBackend, x), attention_backend_names))
        self.transformer = nn.TransformerEncoder(
            transformer_layer, 
            num_layers=self.params.transformer['layers_count'],
            enable_nested_tensor=False, # True is incompatible with layers where norm_first=True
            norm=nn.LayerNorm(self.params.d_model), # output normalizer
        )
        self.init_transformer_weights(self.transformer)

        self.value_head = self.init_weights(nn.Linear(self.params.d_model, 1), 1)
        self.actor_head = self.init_weights(nn.Linear(self.params.d_model, self.params.actions_count), np.sqrt(0.01))
        predictor_head_params = PredictorHead.Params(
            d_model=self.params.d_model,
            actions_count=self.params.actions_count,
        )
        self.predictor_head = PredictorHead(predictor_head_params) 

    @property
    def grad_norm_groups(self):
        d = dict(
            temporal_encoding=[self.temporal_encoding],
            action_plan_encoding=[self.action_plan_encoding],
            transformer=list(self.transformer.parameters()),
            value_head=list(self.value_head.parameters()),
            actor_head=list(self.actor_head.parameters()),
            predictor_head=list(self.predictor_head.parameters()),
        )

        return d

    def train(self, mode=True):
        # DO NOT call super().train(mode)!!!
        # This is due to issues with strange transformer behavior when: eval mode + torch.no_grad() + attention_mask.
        # See: https://share.google/aimode/lFz0O2Qgduhc185Zt
        pass
            
    ForwardResult = namedtuple('ForwardResult', 'actions, action_log_probs, action_plan_logits, action_entropies, values, pred_ob_latents')

    def forward(self, padding_masks, ob_latents, actions=None, predict=False):
        # padding_masks.shape: (batch, seq)
        # ob_latents.shape = (batch, seq, d_model)
        # actions.shape: (batch,)
        batch_size = len(ob_latents)

        # LABEL ob_latents with temporal markers
        ob_latents = ob_latents + self.temporal_encoding
        
        # ATTACH action plan latents to ob_latents
        action_plan_latents = self.action_plan_encoding.expand(batch_size, -1, -1)
        x = torch.cat([ob_latents, action_plan_latents], dim=1) 
        assert x.shape == (batch_size, self.internal_sequence_length, self.params.d_model), x.shape

        # ALIGN padding masks with x's shape
        # Action plan latents are not pad masked so add zeros to padding_masks
        inner_latents_padding_masks = torch.zeros(batch_size, action_plan_latents.shape[1], dtype=padding_masks.dtype, device=padding_masks.device)
        padding_masks = torch.cat([padding_masks, inner_latents_padding_masks], dim=1)
        assert padding_masks.shape == (batch_size, self.internal_sequence_length), padding_masks.shape

        # TRANSFORM
        with sdpa_kernel(self.attention_backend):
            y = self.transformer(
                x, 
                src_key_padding_mask=padding_masks,
                mask=self.attention_mask, 
                is_causal=False, # custom mask is used which is not a standard causal mask 
            )

        assert y.shape == (batch_size, self.internal_sequence_length, self.params.d_model), y.shape

        # ACTOR HEAD
        action_plan_latents = y[:,-action_plan_latents.shape[1]:]
        assert action_plan_latents.shape == (batch_size, self.params.action_plan_length, self.params.d_model)
        
        # Map to logits -> [batch, action_plan_len, action_logits]
        action_plan_logits = self.actor_head(action_plan_latents)
        action_dist = Categorical(logits=action_plan_logits[:,0])
    
        if actions is None:
            # Rollout case
            actions = action_dist.sample() # (batch,)

        assert actions.shape == (batch_size,)
        
        action_log_probs = action_dist.log_prob(actions) # (batch,)
        assert action_log_probs.shape == (batch_size,)
    
        # Evaluate entropies of each item within action plans ("creative sandbox" branch), this way we can control diversity of generated action plans
        action_plan_log_probs = torch.log_softmax(action_plan_logits, dim=-1) # log_softmax == Categorical(logits=logits).log_prob(actions)
        action_plan_probs = torch.softmax(action_plan_logits, dim=-1)
        action_plan_entropies = -torch.sum(action_plan_probs * action_plan_log_probs, dim=-1) # [batch, action_plan_len]

        # VALUE HEAD
        current_state_latents = y[:,self.params.sequence_length-1]
        assert current_state_latents.shape == (batch_size, self.params.d_model)
        values = self.value_head(current_state_latents).squeeze(-1)
        assert values.shape == (batch_size,)

        # PREDICTOR HEAD
        pred_ob_latents = lu.when(
            predict,
            lambda: self.predictor_head(current_state_latents, actions),
            None
        )
        
        return Agent.ForwardResult(
            actions=actions,
            action_log_probs=action_log_probs,
            action_plan_logits=action_plan_logits,
            action_entropies=action_plan_entropies,
            values=values, 
            pred_ob_latents=pred_ob_latents,
        )

    def generate_attention_mask(self):
        # Blank slate matrix to put attention mask components in
        am = torch.zeros(self.internal_sequence_length, self.internal_sequence_length)

        # Ob latents could talk to each other in an ordinary casual way
        obs_am = nn.Transformer.generate_square_subsequent_mask(self.params.sequence_length)
        obs_am = torch.where(torch.isinf(obs_am), 0, 1)
        am[:len(obs_am),:len(obs_am)] = obs_am # top-left square - attention mask for ob latents

        # Action plan latents talk to each other in an ordinary causal way
        ap_am = nn.Transformer.generate_square_subsequent_mask(self.params.action_plan_length)
        ap_am = torch.where(torch.isinf(ap_am), 0, 1)
        am[len(obs_am):,len(obs_am):] = ap_am # bottom-right square

        # Action plan latents could talk to any ob latnets
        am[len(obs_am):,:len(obs_am)] = 1 # bottom-left rect
        assert torch.all(am[:len(obs_am),len(obs_am):] == 0), 'Ob latents should not attend any non-ob latents'
        return torch.where(am == 0, -torch.inf, 0) # nn.TransformerEncoder expectations: 0 - could attend, -inf cannot attend

    @staticmethod
    def init_weights(l, gain=np.sqrt(2), bias_const=0.0):
        '''
        CleanRL style orthogonal initialization helper
        https://share.google/aimode/BDty0f9ZJZ6OXBFFQ
        '''
        
        nn.init.orthogonal_(l.weight, gain=gain)
        
        if l.bias is not None:
            nn.init.constant_(l.bias, bias_const)
            
        return l

    @staticmethod
    def init_transformer_weights(t):
        '''
        Applies orthogonal initialization to the inner transformer blocks.
        https://share.google/aimode/BDty0f9ZJZ6OXBFFQ
        '''
        
        for name, param in t.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                # Appling gain=1.0 keeps variance stable across depth
                nn.init.orthogonal_(param, gain=1.0)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)

## Test

### Basics

In [32]:
# @launchit.disable
t = lu.ScopedVars()
t.device = CONFIG.cuda_device
# t.device = 'cpu'

t.ap = Agent.Params(
    sequence_length=4,
    actions_count=6,
    action_plan_length=10,
    d_model=256,
    transformer=dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH']),
    
)
t.agent = Agent(t.ap).to(t.device)
print(t.agent)
t.params_count = sum(p.numel() for p in t.agent.parameters())
print(f'{t.params_count=:_}')

t.agent.grad_norm_groups;

t.envs_count = 2
t.steps_count = 10

t.ob_latents = torch.zeros((t.envs_count, t.ap.sequence_length, t.ap.d_model)).to(t.device)
t.pmasks = torch.ones((t.envs_count, t.ap.sequence_length)).to(t.device)
t.pmasks[:,-1] = 0

print(f'{t.ob_latents.shape=}')
print(f'{t.pmasks.shape=}')

with torch_utils.eval_guard(t.agent), torch.no_grad():
    r = t.agent(
        padding_masks=t.pmasks,
        ob_latents=t.ob_latents,
    )
    
shape = einops.parse_shape(r.actions, 'e')
assert shape['e'] == t.envs_count
print(f'{r.actions.shape=}, {shape=}')

shape = einops.parse_shape(r.action_log_probs, 'e')
assert shape['e'] == t.envs_count
print(f'{r.action_log_probs.shape=}, {shape=}')

shape = einops.parse_shape(r.action_plan_logits, 'e a l')
assert shape['e'] == t.envs_count
assert shape['a'] == t.ap.action_plan_length
assert shape['l'] == t.ap.actions_count
print(f'{r.action_plan_logits.shape=}, {shape=}')

shape = einops.parse_shape(r.action_entropies, 'e a')
assert shape['e'] == t.envs_count
assert shape['a'] == t.ap.action_plan_length
print(f'{r.action_entropies.shape=}, {shape=}')

shape = einops.parse_shape(r.values, 'e')
assert shape['e'] == t.envs_count
print(f'{r.values.shape=}, {shape=}')

Agent(
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.0, inplace=False)
        (dropout2): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (value_head): Linear(in_features=256, out_features=1, bias=True)
  (actor_head): Linear(in_features=256, out_features=6, bias=True)
  (predictor_head): PredictorHead(
    (action_em

### Quick play

In [33]:
# @launchit.disable
t = lu.ScopedVars()
t.sequence_length = 10
t.device = CONFIG.cuda_device
# t.device = 'cpu'

t.d_model = 256
t.ob_preprocessor = ob_preprocessor.ObPreprocessor(ob_shape=(3, 89, 76), cast_to_float=True)
t.encoder = nn.Sequential(
    nn.Flatten(-3, -1),
    nn.Linear(math.prod(t.ob_preprocessor.ob_shape), t.d_model)
).to(t.device)
t.ap = Agent.Params(
    sequence_length=t.sequence_length,
    actions_count=6,
    action_plan_length=10,
    d_model=t.d_model,
    transformer=dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH']),
)
t.agent = Agent(t.ap).to(t.device)

t.env_step = 0
t.env = create_envs(1)
t.reset_options = {}
t.step_obs, _ = t.env.reset(seed=HP.general.random_seed, options=t.reset_options)
t.step_obs = torch.tensor(t.step_obs.squeeze(0)).to(t.device) 
t.step_obs = t.ob_preprocessor(t.step_obs)
t.step_ob_latents = t.encoder(t.step_obs)
t.ob_latents = torch.zeros(t.sequence_length, t.ap.d_model).to(t.device)
t.ob_latents[-1] = t.step_ob_latents
t.pmasks = torch.ones(t.sequence_length).to(t.device)
t.pmasks[-1] = 0
t.rewards_sum = 0

t.rollout_steps_count = 1000

# verification vars
t.vrf_ob_latents = deque(maxlen=t.sequence_length)
t.vrf_ob_latents.append(t.step_ob_latents.unsqueeze(0))

t.vrf_pmasks = []

for i in range(t.rollout_steps_count):
    t.vrf_pmask = torch.ones(t.sequence_length)
    t.vrf_pmask[-(i+1):] = 0
    t.vrf_pmasks.append(t.vrf_pmask)

t.vrf_pmasks = torch.vstack(t.vrf_pmasks).to(t.device)

with torch.no_grad():
    for t.rs in tqdm(range(0, t.rollout_steps_count)): 
        # verification of FIFO logic correcteness
        t.vrf_ob_latents_as_tensor = torch.cat(list(t.vrf_ob_latents), dim=0)
        assert torch.all(t.ob_latents[~t.pmasks.bool()] == t.vrf_ob_latents_as_tensor)
        assert torch.all(t.pmasks == t.vrf_pmasks[t.env_step])

        # Each call takes 2-3 ms with GPU, so throughput is limited to 1000/2.5 = 400 it/s
        t.agent_result = t.agent(
            ob_latents=t.ob_latents.unsqueeze(0),
            padding_masks=t.pmasks.unsqueeze(0),
        )

        t.action = t.agent_result.actions[0].cpu().unsqueeze(0).numpy()
        # t.action[:] = 1

        t.step_obs, t.reward, t.terminated, t.truncated, t.info = t.env.step(t.action)
        t.step_obs = torch.tensor(t.step_obs.squeeze(0)).to(t.device) # squeeze(1) - squeeze dummy stack frame dim, squeeze(0) - squeeze dummy env dim
        t.step_obs = t.ob_preprocessor(t.step_obs)
        t.step_ob_latents = t.encoder(t.step_obs)
        
        # FIFO logic - push step_obs into tail, stale would fall out (and forget) from the head
        t.pmasks = t.pmasks.roll(shifts=-1, dims=0)
        t.pmasks[-1] = 0
        t.ob_latents = t.ob_latents.roll(shifts=-1, dims=0)
        t.ob_latents[-1] = t.step_ob_latents

        t.rewards_sum += t.reward 

        t.vrf_ob_latents.append(t.step_ob_latents.unsqueeze(0))

        if t.info['is_life_lost'][0]:
            t.r = t.info['life_returns'][0].item()
            t.l = t.info['life_lengths'][0].item()
            LOG(f'{t.env_step:03} Life lost: r={t.r:6}, l={t.l:6}, rsum={t.rewards_sum}')
            # LOG(f'{t.env_step:03} Life lost: r={t.r:6}, l={t.l:6}; {t.info=}')

        if t.info['is_episode_over'][0]:
            t.r = t.info['episode_returns'][0].item()
            t.l = t.info['episode_lengths'][0].item()
            LOG(f'{t.env_step:03} EPISODE OVER: r={t.r:6}, l={t.l:6}, rsum={t.rewards_sum}')
            # LOG(f'{t.env_step:03} EPISODE OVER: r={t.r:6}, l={t.l:6}; {t.info=}')

        if t.terminated or t.truncated:
            # t.reset_options = dict(ram_patches={-1: EnvRamPatcher()(['last_life', 'temperature_10'])})
            t.reset_options = {}
            t.step_obs, _ = t.env.reset(options=t.reset_options)
            t.step_obs = torch.tensor(t.step_obs.squeeze(0)).to(t.device)
            t.step_obs = t.ob_preprocessor(t.step_obs)
            t.step_ob_latents = t.encoder(t.step_obs)
            
            t.ob_latents.zero_()
            t.ob_latents[-1] = t.step_ob_latents
            t.pmasks.fill_(1)
            t.pmasks[-1] = 0
            
            t.vrf_ob_latents.clear()
            t.vrf_ob_latents.append(t.step_ob_latents.unsqueeze(0))
            
            t.env_step = 0
        else:
            assert not 'episode' in t.info
            assert not 'is_game_over' in t.info
            t.env_step += 1

  0%|          | 0/1000 [00:00<?, ?it/s]

077 Life lost: r=  10.0, l=   312, rsum=[1]
209 Life lost: r=  40.0, l=   528, rsum=[5]
283 Life lost: r=   0.0, l=   296, rsum=[5]
337 Life lost: r=   0.0, l=   216, rsum=[5]
337 EPISODE OVER: r=  50.0, l=  1353, rsum=[5]
147 Life lost: r=  60.0, l=   592, rsum=[11]
231 Life lost: r=  10.0, l=   336, rsum=[12]
308 Life lost: r=   0.0, l=   308, rsum=[12]
412 Life lost: r=  60.0, l=   415, rsum=[18]
412 EPISODE OVER: r= 130.0, l=  1663, rsum=[18]
070 Life lost: r=   0.0, l=   284, rsum=[18]
147 Life lost: r=  10.0, l=   308, rsum=[19]


### VectorEnv

In [34]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 2
t.sequence_length = 10
t.device = CONFIG.cuda_device
# t.device = 'cpu'

t.d_model = 256
t.ob_preprocessor = ob_preprocessor.ObPreprocessor(ob_shape=(3, 89, 76), cast_to_float=True)
t.encoder = nn.Sequential(
    nn.Flatten(-3, -1),
    nn.Linear(math.prod(t.ob_preprocessor.ob_shape), t.d_model)
).to(t.device)
t.ap = Agent.Params(
    sequence_length=t.sequence_length,
    actions_count=6,
    action_plan_length=10,
    d_model=t.d_model,
    transformer=dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH']),
)
t.agent = Agent(t.ap).to(t.device)

t.env_steps = torch.zeros(t.envs_count).int()
t.envs = create_envs(t.envs_count)
t.step_obs, _ = t.envs.reset(seed=HP.general.random_seed)
t.step_obs = torch.tensor(t.step_obs).to(t.device)
t.step_obs = t.ob_preprocessor(t.step_obs)
t.step_ob_latents = t.encoder(t.step_obs)
t.ob_latents = torch.zeros((t.envs_count, t.sequence_length, t.ap.d_model)).to(t.device)
t.ob_latents[:,-1] = t.step_ob_latents
t.pmasks = torch.ones((t.envs_count, t.sequence_length)).to(t.device)
t.pmasks[:,-1] = 0
t.rewards_sums = np.zeros(t.envs_count)

t.rollout_steps_count = 1000
t.life_lost_r_sum = 0
t.episode_over_r_sum = 0

with torch.no_grad():
    for t.rs in tqdm(range(0, t.rollout_steps_count)): 
        # Each call takes 2-3 ms with GPU, so throughput is limited to 1000/2.5 = 400 it/s
        t.agent_result = t.agent(
            ob_latents=t.ob_latents,
            padding_masks=t.pmasks,
        )

        t.step_obs, t.rewards, t.terminations, t.truncations, t.infos = t.envs.step(t.agent_result.actions.cpu().numpy())
        t.step_obs = torch.tensor(t.step_obs).to(t.device)
        t.step_obs = t.ob_preprocessor(t.step_obs)
        t.step_ob_latents = t.encoder(t.step_obs)
        
        # FIFO logic - push ob latents into tail, stale would fall out (and forget) from the head
        t.pmasks = t.pmasks.roll(shifts=-1, dims=1)
        t.pmasks[:,-1] = 0
        t.ob_latents = t.ob_latents.roll(shifts=-1, dims=1)
        t.ob_latents[:,-1] = t.step_ob_latents

        assert t.rewards_sums.shape == t.rewards.shape
        t.rewards_sums += t.rewards

        if 'is_life_lost' in t.infos:
            for t.env_ind in np.argwhere(t.infos['is_life_lost']).ravel():
                t.r = t.infos['life_returns'][t.env_ind]
                t.l = t.infos['life_lengths'][t.env_ind]
                t.rs = t.rewards_sums[t.env_ind]
                LOG(f'{t.env_steps[t.env_ind]:03} ENV:{t.env_ind} Life lost: r={t.r:6}, l={t.l:6}, rs={t.rs:.2f}, lives={t.infos['lives'][t.env_ind]}')
                t.life_lost_r_sum += t.r

        if 'is_episode_over' in t.infos:
            for t.env_ind in np.argwhere(t.infos['is_episode_over']).ravel():
                t.r = t.infos['episode_returns'][t.env_ind]
                t.l = t.infos['episode_lengths'][t.env_ind]
                t.rs = t.rewards_sums[t.env_ind]
                LOG(f'{t.env_steps[t.env_ind]:03} ENV:{t.env_ind} EPISOVE OVER: r={t.r:6}, l={t.l:6}, rs={t.rs:.2f}, lives={t.infos['lives'][t.env_ind]}')
                t.episode_over_r_sum += t.r

        t.done_envs = np.logical_or(t.terminations, t.truncations)
            
        for t.env_ind, t.is_done in enumerate(t.done_envs):
            if not t.is_done:
                t.env_steps[t.env_ind] += 1
            else:
                # t.step_obs[t.env_ind] is a death screen, handle it properly, e.g. preserve in a rollout buffer
                t.pmasks[t.env_ind] = 1
                t.pmasks[t.env_ind,-1] = 0
                t.env_steps[t.env_ind] = 0

        if np.any(t.done_envs):
            t.reset_obs, t.reset_infos = t.envs.reset(options=dict(reset_mask=t.done_envs))
            LOG(f'{t.done_envs=}, {t.reset_infos=}')

            for t.env_ind in np.argwhere(t.done_envs).ravel():
                assert t.reset_infos['lives'][t.env_ind] > t.infos['lives'][t.env_ind]
                assert t.reset_infos['episode_frame_number'][t.env_ind] <= t.infos['episode_frame_number'][t.env_ind]

            for t.env_ind in np.argwhere(~t.done_envs).ravel():
                assert t.reset_infos['lives'][t.env_ind] == t.infos['lives'][t.env_ind]
                assert t.reset_infos['episode_frame_number'][t.env_ind] == t.infos['episode_frame_number'][t.env_ind]
            
            t.reset_obs = torch.tensor(t.reset_obs[t.done_envs]).to(t.device)
            t.reset_obs = t.ob_preprocessor(t.reset_obs)
            t.old_ob_latents = t.step_ob_latents[~t.done_envs].clone()
            t.step_ob_latents[t.done_envs] = t.encoder(t.reset_obs)
            assert torch.all(t.old_ob_latents == t.step_ob_latents[~t.done_envs]) # ensure we didn't touch envs which are not done

print(f'{t.life_lost_r_sum=}, {t.episode_over_r_sum=}')

  0%|          | 0/1000 [00:00<?, ?it/s]

115 ENV:1 Life lost: r=  20.0, l=   464, rs=2.00, lives=3
135 ENV:0 Life lost: r=  60.0, l=   544, rs=6.00, lives=3
250 ENV:0 Life lost: r=  10.0, l=   460, rs=7.00, lives=2
252 ENV:1 Life lost: r=  10.0, l=   548, rs=3.00, lives=2
339 ENV:0 Life lost: r=  20.0, l=   356, rs=9.00, lives=1
384 ENV:0 Life lost: r=   0.0, l=   180, rs=9.00, lives=0
384 ENV:0 EPISOVE OVER: r=  90.0, l=  1541, rs=9.00, lives=0
t.done_envs=array([ True, False]), t.reset_infos={'env_id': array([0, 1], dtype=int32), 'lives': array([4, 2], dtype=int32), 'levels_passed': array([0, 0], dtype=int32), 'frame_number': array([1553, 1549], dtype=int32), 'episode_frame_number': array([  12, 1549], dtype=int32)}
394 ENV:1 Life lost: r=  60.0, l=   568, rs=9.00, lives=1
446 ENV:1 Life lost: r=  10.0, l=   205, rs=10.00, lives=0
446 ENV:1 EPISOVE OVER: r= 100.0, l=  1794, rs=10.00, lives=0
t.done_envs=array([False,  True]), t.reset_infos={'env_id': array([0, 1], dtype=int32), 'lives': array([4, 4], dtype=int32), 'levels_p

### Batched FIFO logic

In [35]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 2
t.sequence_length = 10
t.d_model = 256
t.rollout_steps_count = 100

t.env_steps = torch.zeros(t.envs_count, dtype=torch.long)
t.env_inds = torch.arange(t.envs_count)
t.ob_latents = torch.zeros(t.envs_count, t.sequence_length, t.d_model)
t.pmasks = torch.ones((t.envs_count, t.sequence_length))

# Verification vars
t.vrf_ob_latents = torch.zeros(t.envs_count, t.sequence_length, t.d_model)
t.vrf_pmasks = torch.ones(t.envs_count, t.sequence_length)

for i in tqdm(range(t.rollout_steps_count)):
    t.new_ob_latents = torch.rand(t.envs_count, t.d_model)

    # Serial version of FIFO logic
    for j in range(t.envs_count):
        t.vrf_ob_latents[j] = t.vrf_ob_latents[j].roll(shifts=-1, dims=0)
        t.vrf_ob_latents[j,-1] = t.new_ob_latents[j]
        
        t.vrf_pmasks[j] = t.vrf_pmasks[j].roll(shifts=-1, dims=0)
        t.vrf_pmasks[j,-1] = 0

    # Batched version of FIFO logic
    t.ob_latents = t.ob_latents.roll(shifts=-1, dims=1)
    t.ob_latents[:,-1] = t.new_ob_latents

    t.pmasks = t.pmasks.roll(shifts=-1, dims=1)
    t.pmasks[:,-1] = 0

    # Serial and batched must produce the same results
    assert torch.all(t.ob_latents == t.vrf_ob_latents)
    assert torch.all(t.pmasks == t.vrf_pmasks)

    for t.env_ind, t.done in enumerate(torch.rand(t.envs_count) < 0.05):
        if t.done:
            t.ob_latents[t.env_ind].zero_()
            t.pmasks[t.env_ind].fill_(1)

            t.vrf_ob_latents[t.env_ind].zero_()
            t.vrf_pmasks[t.env_ind].fill_(1)
            LOG(f'Rollout step {i}, env {t.env_ind} game over, restarting')

    t.env_steps += 1

  0%|          | 0/100 [00:00<?, ?it/s]

Rollout step 5, env 0 game over, restarting
Rollout step 15, env 0 game over, restarting
Rollout step 16, env 0 game over, restarting
Rollout step 26, env 1 game over, restarting
Rollout step 29, env 0 game over, restarting
Rollout step 30, env 1 game over, restarting
Rollout step 31, env 1 game over, restarting
Rollout step 51, env 0 game over, restarting
Rollout step 51, env 1 game over, restarting
Rollout step 54, env 1 game over, restarting
Rollout step 62, env 1 game over, restarting
Rollout step 84, env 1 game over, restarting
Rollout step 96, env 0 game over, restarting


### Ob latents reconstruction

In [36]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 2
t.sequence_length = 10
t.d_model = 256
t.rollout_steps_count = 100

t.env_inds = torch.arange(t.envs_count)
t.ob_latents_storage = torch.zeros(t.envs_count, t.rollout_steps_count + 1, t.d_model)
t.ob_latents = torch.zeros(t.envs_count, t.sequence_length, t.d_model)
t.ob_latents_inds = torch.zeros((t.envs_count, t.sequence_length)).long()

for t.rollout_step in tqdm(range(t.rollout_steps_count)):
    t.new_ob_latents = torch.rand(t.envs_count, t.d_model)

    t.ob_latents_storage[:,t.rollout_step+1] = t.new_ob_latents

    t.ob_latents = t.ob_latents.roll(shifts=-1, dims=1)
    t.ob_latents[:,-1] = t.new_ob_latents
    
    t.ob_latents_inds = t.ob_latents_inds.roll(shifts=-1, dims=1)
    t.ob_latents_inds[:,-1] = t.rollout_step + 1

    t.dones = torch.rand(2) < 0.05

    for t.env_ind, t.done in enumerate(t.dones):
        if t.done:
            t.ob_latents[t.env_ind].zero_()
            t.ob_latents_inds[t.env_ind].zero_()

    t.ob_latents_inds_ex = einops.rearrange(t.ob_latents_inds, 'e m -> e m 1')
    t.ob_latents_inds_ex = t.ob_latents_inds_ex.expand((-1, -1, t.d_model))
    t.reconstr_ob_latents = torch.gather(t.ob_latents_storage, dim=1, index=t.ob_latents_inds_ex)

    assert torch.all(t.ob_latents == t.reconstr_ob_latents)

  0%|          | 0/100 [00:00<?, ?it/s]

## Configure

In [37]:
# @launchit.disable
# @launchit.collect
HP.agent.parent = None 
HP.agent.sequence_length = 4
HP.agent.action_plan_length = 10 
HP.agent.d_model = 256 
HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
HP.agent.is_trainable = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': 'train',
 'launch_id': 0,
 'general': {'comment': None,
             'random_seed': 48,
             'is_torch_deterministic': True,
             'is_torch_compile': False,
             'is_torch_amp': True},
 'env': {'count': 32, 'is_episodic_life': True, 'actions_count': 6},
 'vision_head': {'parent': {'model': '18d_world_model_09:40',
                            'weights': '18d_world_model_09:40'},
                 'is_trainable': False},
 'encoder': {'parent': {'model': '18d_world_model_09:40',
                        'weights': '18d_world_model_09:40'},
             'is_trainable': True},
 'agent': {'parent': None,
           'sequence_length': 10,
           'action_plan_length': 10,
           'd_model': 256,
           'transformer': {'layers_count': 3,
                           'heads_count': 4,
                           'attention_backend': ['EFFICIENT_ATTENTION',
                                                 'MATH']},
           'is_trainable': True},
 '

## Initrd

In [38]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig':
    if HP.agent.parent:
        coords = hp_parse_artifact_source(HP.agent.parent)
        CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='json', asset_classifier='agent_params', maven_group_id=coords.group_id)
        CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='pt', asset_classifier='agent', maven_group_id=coords.group_id)
# @launchit.stop

## Create

In [39]:
ap = Agent.Params(
    sequence_length=HP.agent.sequence_length,
    actions_count=RT.env_action_space_shape[0],
    action_plan_length=HP.agent.action_plan_length,
    d_model=HP.agent.d_model,
    transformer=HP.agent.transformer,
)
RT.agent = Agent(ap).to(CONFIG.cuda_device)
assert RT.encoder.params.d_model == RT.agent.params.d_model
LOG(f'Agent created')

if HP.agent.parent is not None:
    coords = hp_parse_artifact_source(HP.agent.parent)
    agent_weights_data = RT.artifact_registry.get_asset_content(
        coords.model_name, 
        coords.model_version, 
        asset_ext='pt', 
        asset_classifier='agent', 
        maven_group_id=coords.group_id,
    )

    with io.BytesIO(agent_weights_data) as b:
        kwargs = {}
        
        if not CONFIG.is_cuda:
            kwargs['map_location'] = torch.device('cpu')
            
        agent_state_dict = torch.load(b, **kwargs)
        RT.agent.load_state_dict(agent_state_dict)

    LOG(f'Agent state loaded from "{HP.agent.parent}"')

if HP.general.is_torch_compile:
    RT.agent = torch.compile(RT.agent, fullgraph=True)
    LOG(f'Agent compiled')

RT.agent.is_trainable = HP.agent.is_trainable
assert next(iter(RT.agent.parameters())).requires_grad == HP.agent.is_trainable
LOG(f'{RT.agent.is_trainable=}')

Agent created
RT.agent.is_trainable=True


# Dataset

## Dataset

In [40]:
@dataclass(slots=True)
class Dataset:
    # METADATA
    sequence_length: int = None
    rollout_steps_count: int = None
    prologue_size: int = None
    action_plan_length: int = None
    actions_count: int = None

    # MAIN DATA
    # Agent's input data
    pmasks: object = None # padding masks
    obs: object = None
    ob_latents: object = None
    
    # Agent's output data
    actions: object = None
    action_log_probs: object = None
    values: object = None
    next_action_plan_logits: object = None
    
    # Environment response
    next_obs: object = None # observation which agent received after stepping (used to learn predictions)
    next_ob_latents: object = None # latent of next_obs
    dones: object = None
    advantages: object = None
    returns: object = None

    # Indices
    env_inds: object = None
    step_inds: object = None
    
    # ITERATION SUPPORT
    valid_inds: object = None
    batch_size: int = None
    is_shuffled: bool = None
    iter_inds: object = None
    iter_pos: int = None
    flattened: tuple = None
    selected: tuple = None
    bw_inds_stem: object = None

    Batch = namedtuple('Batch', 
                       'pmasks, obs, ob_latents, ' + 
                       'actions, action_log_probs, values, next_action_plan_logits, next_obs, next_ob_latents, ' + 
                       'dones, advantages, returns, ' + 
                       'env_inds, step_inds, b_inds')
    
    def __init__(
        self,
        envs_count,
        ob_shape,
        d_model, 
        rollout_steps_count, 
        sequence_length,
        action_plan_length,
        actions_count,
    ):
        self.sequence_length = sequence_length
        self.rollout_steps_count = rollout_steps_count
        self.prologue_size = (sequence_length - 1)
        rollout_buf_size = self.prologue_size + self.rollout_steps_count # prologue (mem window from prev rollout) + actual rollout data
        assert rollout_buf_size > 0
        self.action_plan_length = action_plan_length
        self.actions_count = actions_count
        
        # Agent's input data
        self.pmasks = torch.zeros(envs_count, rollout_buf_size, sequence_length, device=CONFIG.cuda_device) 
        self.obs = torch.zeros((envs_count, rollout_buf_size, *ob_shape), dtype=torch.uint8, device=CONFIG.cuda_device)
        self.ob_latents = torch.zeros(envs_count, rollout_buf_size, d_model, device=CONFIG.cuda_device)

        # Agent's output data
        self.actions = torch.zeros((envs_count, rollout_buf_size), dtype=torch.long, device=CONFIG.cuda_device)
        self.action_log_probs = torch.zeros((envs_count, rollout_buf_size), device=CONFIG.cuda_device)
        self.values = torch.zeros((envs_count, rollout_buf_size), device=CONFIG.cuda_device)
        self.next_action_plan_logits = torch.zeros((envs_count, rollout_buf_size, action_plan_length, actions_count), device=CONFIG.cuda_device)

        # Environment response
        self.next_obs = torch.zeros((envs_count, rollout_buf_size, *ob_shape), dtype=torch.uint8, device=CONFIG.cuda_device)
        self.next_ob_latents = torch.zeros(envs_count, rollout_buf_size, d_model, device=CONFIG.cuda_device)
        self.dones = torch.zeros((envs_count, rollout_buf_size), device=CONFIG.cuda_device)
        self.advantages = torch.zeros((envs_count, rollout_buf_size), device=CONFIG.cuda_device)
        self.returns = torch.zeros((envs_count, rollout_buf_size), device=CONFIG.cuda_device)

        # a-la meshgrid to easily get pairs (env_ind, step_ind) by b_inds
        self.env_inds = torch.arange(envs_count, device=CONFIG.cuda_device).unsqueeze(1).expand(-1, rollout_buf_size)
        self.step_inds = torch.arange(-self.prologue_size, self.rollout_steps_count, device=CONFIG.cuda_device).unsqueeze(0).expand(envs_count, -1)
        assert self.env_inds.shape == self.step_inds.shape, (self.env_inds.shape, self.step_inds.shape)

    @property
    def main_data(self):
        return dict(
            pmasks=self.pmasks,
            obs=self.obs,
            ob_latents=self.ob_latents,
            actions=self.actions,
            action_log_probs=self.action_log_probs,
            values=self.values,
            next_action_plan_logits=self.next_action_plan_logits,
            next_obs=self.next_obs,
            next_ob_latents=self.next_ob_latents,
            dones=self.dones,
            advantages=self.advantages,
            returns=self.returns,
            env_inds=self.env_inds,
            step_inds=self.step_inds,
        )

    @property
    def batches_count(self):
        assert self.batch_size is not None, 'init_iteration must be called before querying batches_count'
        envs_count = len(self.ob_latents)
        return envs_count * self.rollout_steps_count // self.batch_size

    @property
    def items_count(self):
        envs_count = len(self.ob_latents)
        return envs_count * self.rollout_steps_count

    def init_iteration(self, is_shuffled, batches_count=None, batch_size=None):
        if batches_count is not None and batch_size is not None:
            assert False, 'Either batches_count or batch_size must be specified, but not both'
        elif batches_count is None and batch_size is None:
            assert False, 'Either batches_count or batch_size must be specified'

        cuda_device = self.pmasks.device
        envs_count = len(self.ob_latents)
        rollout_buf_size = self.ob_latents.shape[1]
        ob_shape = tuple(self.obs.shape[2:])
        d_model = self.ob_latents.shape[-1]
        valid_inds = []
        
        for env_ind in range(envs_count):
            offset = env_ind * rollout_buf_size + self.prologue_size
            indices_for_env = offset + torch.arange(self.rollout_steps_count)
            valid_inds.append(indices_for_env)
            
        self.valid_inds = torch.cat(valid_inds).to(CONFIG.cuda_device)

        if batches_count is not None:
            assert batches_count > 0
            assert envs_count * self.rollout_steps_count % batches_count == 0, (
                f'Not a whole number of batches for given rollout data size: {((envs_count * self.rollout_steps_count) / batches_count)=}'
            )
            self.batch_size = envs_count * self.rollout_steps_count // batches_count
        else:
            assert batch_size > 0
            assert envs_count * self.rollout_steps_count % batch_size == 0, (
                f'Not a whole number of batches for given rollout data size: {((envs_count * self.rollout_steps_count) / batch_size)=}'
            )
            self.batch_size = batch_size
        
        self.is_shuffled = is_shuffled

        self.flattened = Dataset.Batch(
            pmasks=self.pmasks.view(-1, *self.pmasks.shape[2:]),
            obs=self.obs.view(-1, *self.obs.shape[2:]),
            ob_latents=self.ob_latents.view(-1, *self.ob_latents.shape[2:]),
            actions=self.actions.view(-1, *self.actions.shape[2:]),
            action_log_probs=self.action_log_probs.view(-1, *self.action_log_probs.shape[2:]),
            values=self.values.view(-1, *self.values.shape[2:]),
            next_action_plan_logits=self.next_action_plan_logits.view(-1, *self.next_action_plan_logits.shape[2:]),
            next_obs=self.next_obs.view(-1, *self.next_obs.shape[2:]),
            next_ob_latents=self.next_ob_latents.view(-1, *self.next_ob_latents.shape[2:]),
            dones=self.dones.view(-1, *self.dones.shape[2:]),
            advantages=self.advantages.view(-1, *self.advantages.shape[2:]),
            returns=self.returns.view(-1, *self.returns.shape[2:]),
            env_inds=self.env_inds.reshape(-1, *self.env_inds.shape[2:]),
            step_inds=self.step_inds.reshape(-1, *self.step_inds.shape[2:]),
            b_inds=None,
        )
        self.selected = Dataset.Batch(
            pmasks=torch.ones(self.batch_size, self.sequence_length, device=cuda_device), # stored as sequence
            obs=torch.zeros(self.batch_size * self.sequence_length, *ob_shape, dtype=self.obs.dtype, device=cuda_device),
            ob_latents=torch.zeros(self.batch_size * self.sequence_length, d_model, dtype=self.ob_latents.dtype, device=cuda_device),
            actions=torch.zeros(self.batch_size, dtype=self.actions.dtype, device=cuda_device),
            action_log_probs=torch.zeros(self.batch_size, device=cuda_device),
            values=torch.zeros(self.batch_size, device=cuda_device),
            next_action_plan_logits=torch.zeros(self.batch_size, self.action_plan_length, self.actions_count, device=cuda_device),
            next_obs=torch.zeros(self.batch_size, *ob_shape, dtype=self.next_obs.dtype, device=cuda_device),
            next_ob_latents=torch.zeros(self.batch_size, d_model, dtype=self.next_ob_latents.dtype, device=cuda_device),
            dones=torch.zeros(self.batch_size, device=cuda_device),
            advantages=torch.zeros(self.batch_size, device=cuda_device),
            returns=torch.zeros(self.batch_size, device=cuda_device),
            env_inds=torch.zeros(self.batch_size, dtype=self.env_inds.dtype, device=cuda_device),
            step_inds=torch.zeros(self.batch_size, dtype=self.step_inds.dtype, device=cuda_device),
            b_inds=None,
        )
        self.bw_inds_stem = torch.arange(-self.sequence_length + 1, 1, device=cuda_device)
        self.bw_inds_stem = self.bw_inds_stem.unsqueeze(0).expand(self.batch_size, self.bw_inds_stem.shape[0])

    def __iter__(self):
        cuda_device = self.pmasks.device
        
        if self.is_shuffled:
            valids_inds_order = torch.randperm(len(self.valid_inds), device=cuda_device)
            self.iter_inds = self.valid_inds[valids_inds_order]
        else:
            self.iter_inds = self.valid_inds

        self.iter_pos = 0
        return self
    
    def __next__(self):
        if self.iter_pos >= len(self.iter_inds):
            raise StopIteration

        b_inds = self.iter_inds[self.iter_pos:self.iter_pos+self.batch_size]
        self.iter_pos += self.batch_size

        # windows stored as is
        torch.gather(input=self.flattened.pmasks, index=b_inds.unsqueeze(1).expand(-1, self.sequence_length), dim=0, out=self.selected.pmasks)
        # reconstructed windows. With ob_latents main justification is memory limits
        bw_inds = self.bw_inds_stem + b_inds.unsqueeze(1) # batch-window indices, shape: [batch, seq_len]
        bw_inds = bw_inds.view(-1)
        torch.index_select(input=self.flattened.obs, index=bw_inds, dim=0, out=self.selected.obs)
        torch.index_select(input=self.flattened.ob_latents, index=bw_inds, dim=0, out=self.selected.ob_latents)
        # non-windows
        torch.index_select(input=self.flattened.actions, index=b_inds, dim=0, out=self.selected.actions)
        torch.index_select(input=self.flattened.action_log_probs, index=b_inds, dim=0, out=self.selected.action_log_probs)
        torch.index_select(input=self.flattened.values, index=b_inds, dim=0, out=self.selected.values)
        torch.index_select(input=self.flattened.next_action_plan_logits, index=b_inds, dim=0, out=self.selected.next_action_plan_logits)
        torch.index_select(input=self.flattened.next_obs, index=b_inds, dim=0, out=self.selected.next_obs)
        torch.index_select(input=self.flattened.next_ob_latents, index=b_inds, dim=0, out=self.selected.next_ob_latents)
        torch.index_select(input=self.flattened.dones, index=b_inds, dim=0, out=self.selected.dones)
        torch.index_select(input=self.flattened.advantages, index=b_inds, dim=0, out=self.selected.advantages)
        torch.index_select(input=self.flattened.returns, index=b_inds, dim=0, out=self.selected.returns)
        torch.index_select(input=self.flattened.env_inds, index=b_inds, dim=0, out=self.selected.env_inds)
        torch.index_select(input=self.flattened.step_inds, index=b_inds, dim=0, out=self.selected.step_inds)
        
        return Dataset.Batch(
            pmasks=self.selected.pmasks,
            obs=self.selected.obs.view(self.batch_size, self.sequence_length, *self.selected.obs.shape[1:]),
            ob_latents=self.selected.ob_latents.view(self.batch_size, self.sequence_length, *self.selected.ob_latents.shape[1:]),
            actions=self.selected.actions,
            action_log_probs=self.selected.action_log_probs,
            values=self.selected.values,
            next_action_plan_logits=self.selected.next_action_plan_logits.view(self.batch_size, *self.next_action_plan_logits.shape[2:]),
            next_obs=self.selected.next_obs,
            next_ob_latents=self.selected.next_ob_latents,
            dones=self.selected.dones,
            advantages=self.selected.advantages,
            returns=self.selected.returns,
            env_inds=self.selected.env_inds,
            step_inds=self.selected.step_inds,
            b_inds=b_inds,
        )

## Test

### Smoke test

In [41]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 8
t.rollout_steps_count = 64
t.sequence_length = 10
t.action_plan_length = 8
t.actions_count = 6
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=RT.vision_head.params.ob_shape,
    d_model=RT.agent.params.d_model, 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=t.sequence_length, 
    action_plan_length=t.action_plan_length,
    actions_count=t.actions_count,
)
t.dataset.init_iteration(batches_count=2, is_shuffled=False)

for t.name, t.v in t.dataset.main_data.items():
    if t.v is not None:
        LOG(f'{t.name:>38}: {str(t.v.device):>6}, {str(t.v.dtype):>15}, {t.v.shape}')

assert len(t.dataset.valid_inds) == (t.envs_count * t.rollout_steps_count)
assert len(t.dataset.valid_inds.unique()) == (t.envs_count * t.rollout_steps_count)

t.prev_env_valid_inds = None

for t.env_ind in range(t.envs_count):
    t.prev_env_training_ind = lu.when(t.prev_env_valid_inds is not None, lambda: t.prev_env_valid_inds[-1] + 1, 0)
    t.env_valid_inds = t.prev_env_training_ind + (t.sequence_length - 1) + torch.arange(t.rollout_steps_count).to(CONFIG.cuda_device)
    assert torch.all(t.env_valid_inds == t.dataset.valid_inds[t.env_ind*t.rollout_steps_count:(t.env_ind+1)*t.rollout_steps_count])
    t.prev_env_valid_inds = t.env_valid_inds

                                pmasks:    cpu,   torch.float32, torch.Size([8, 73, 10])
                                   obs:    cpu,     torch.uint8, torch.Size([8, 73, 3, 89, 76])
                            ob_latents:    cpu,   torch.float32, torch.Size([8, 73, 256])
                               actions:    cpu,     torch.int64, torch.Size([8, 73])
                      action_log_probs:    cpu,   torch.float32, torch.Size([8, 73])
                                values:    cpu,   torch.float32, torch.Size([8, 73])
               next_action_plan_logits:    cpu,   torch.float32, torch.Size([8, 73, 8, 6])
                              next_obs:    cpu,     torch.uint8, torch.Size([8, 73, 3, 89, 76])
                       next_ob_latents:    cpu,   torch.float32, torch.Size([8, 73, 256])
                                 dones:    cpu,   torch.float32, torch.Size([8, 73])
                            advantages:    cpu,   torch.float32, torch.Size([8, 73])
                       

### Iteration

In [42]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 32
t.rollout_steps_count = 128
t.sequence_length = 10
t.action_plan_length = 8
t.actions_count = 6
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=RT.vision_head.params.ob_shape,
    d_model=RT.agent.params.d_model, 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=t.sequence_length, 
    action_plan_length=t.action_plan_length,
    actions_count=t.actions_count,
)
t.dataset.init_iteration(batches_count=2, is_shuffled=False)

for t.batch in t.dataset:
    pass

for t.fn in t.batch._fields:
    if getattr(t.batch, t.fn) is not None:
        LOG(f'{t.fn:>20}.shape={getattr(t.batch, t.fn).shape}')
    else:
        LOG(f'{t.fn:>20}=None')

              pmasks.shape=torch.Size([2048, 10])
                 obs.shape=torch.Size([2048, 10, 3, 89, 76])
          ob_latents.shape=torch.Size([2048, 10, 256])
             actions.shape=torch.Size([2048])
    action_log_probs.shape=torch.Size([2048])
              values.shape=torch.Size([2048])
next_action_plan_logits.shape=torch.Size([2048, 8, 6])
            next_obs.shape=torch.Size([2048, 3, 89, 76])
     next_ob_latents.shape=torch.Size([2048, 256])
               dones.shape=torch.Size([2048])
          advantages.shape=torch.Size([2048])
             returns.shape=torch.Size([2048])
            env_inds.shape=torch.Size([2048])
           step_inds.shape=torch.Size([2048])
              b_inds.shape=torch.Size([2048])


# Rollout

## RolloutEnvStorySampler

In [43]:
class RolloutEnvStorySampler:
    def __init__(self, rams, ram_patches, stories):
        assert rams is None or isinstance(rams, dict)
        assert ram_patches is None or isinstance(ram_patches, list)
        assert isinstance(stories, list)
        assert stories
        self.rams = rams
        self.ram_patches = ram_patches
        self.ram_patcher = EnvRamPatcher()
        self.stories = stories

        if self.ram_patches is not None:
            # Verify for all atomic ram_patch there is a ram_patcher func
            for atomic_ram_patch in set(itertools.chain.from_iterable(self.ram_patches)):
                getattr(self.ram_patcher, atomic_ram_patch)
 
    EnvStory = namedtuple('EnvStory', 'ram, ram_patch, desc')

    def sample(self):
        desc = []
        
        story = RNG.choice(self.stories).item()
        avl_rams, avl_ram_patches = story.split(';')
        assert avl_rams, story
        assert avl_ram_patches, story

        if self.rams is None or not self.rams:
            ram = None
            desc.append('None')
        else:
            ram = self.pick(list(self.rams.keys()), avl_rams)
            desc.append(str(ram))
            ram = self.rams[ram]

        ram_patch = self.pick(self.ram_patches, avl_ram_patches)

        if ram_patch is None or not ram_patch:
            ram_patch = None
            desc.append('None')
        else:
            desc.append('+'.join(ram_patch))

        return RolloutEnvStorySampler.EnvStory(ram=ram, ram_patch=ram_patch, desc='/'.join(desc))

    @staticmethod
    def pick(l, inds):
        if l is None or not l:
            return None
            
        subl = None
        
        if inds == '*':
            subl = l
        elif ':' in inds:
            slice_inds = inds.split(':')
            assert len(slice_inds) == 2, inds
            subl = l[slice(int(slice_inds[0]), int(slice_inds[1]))]
        else:
            enum_inds = inds.split(',')
            subl = []
            
            for ind in enum_inds:
                subl.append(l[int(ind)])

        i = RNG.choice(len(subl))
        return subl[i]

## RolloutManager

In [44]:
class RolloutManager:
    def __init__(self, vision_head, encoder, agent, envs_count, random_seed, is_episodic_life=True, device=None):
        self.preprocess_obs = ob_preprocessor.ObPreprocessor(vision_head.params.ob_shape, cast_to_float=False)
        self.vision_head = vision_head
        self.encoder = encoder
        self.agent = agent
        self.envs_count = envs_count
        self.env_inds = torch.arange(envs_count)
        self.random_seed = random_seed
        self.is_episodic_life = is_episodic_life
        self.device = lu.coalesce(device, CONFIG.cuda_device)

        # Sliding windows
        sequence_length = self.agent.params.sequence_length
        self.w_pmasks = torch.ones(envs_count, sequence_length).to(self.device)
        self.w_obs = torch.zeros(envs_count, sequence_length, *self.vision_head.params.ob_shape, dtype=torch.uint8).to(self.device)
        self.w_ob_latents = torch.zeros(envs_count, sequence_length, self.agent.params.d_model).to(self.device)
        
        thread_pool_size = lu.coalesce_fn(os.environ.get('ALE_THREAD_POOL_SIZE'), int, None)
        thread_affinity_offset = lu.coalesce_fn(os.environ.get('ALE_THREAD_AFFINITY_OFFSET'), int, None)
        self.envs = create_envs(envs_count, thread_pool_size=thread_pool_size, thread_affinity_offset=thread_affinity_offset)
        LOG(f'Envs created: {envs_count} envs, {thread_pool_size=}, {thread_affinity_offset=}', when=not CONFIG.is_interactive)

        self.env_ram_patcher = EnvRamPatcher()
        self.is_reset = False
        self.story_counters = defaultdict(int)
        self.levels_passed_counters = np.zeros(self.envs_count, dtype=np.int32)

    def reset(self, dataset, rollout_env_story_sampler=None):
         with torch_utils.eval_mode(self.vision_head), torch_utils.eval_mode(self.encoder), torch_utils.eval_mode(self.agent), torch.no_grad():
             reset_options = self.get_rams_and_ram_patches(rollout_env_story_sampler, self.env_inds)
             self.obs, _ = self.envs.reset(seed=self.random_seed, options=reset_options)
             self.obs = torch.tensor(self.obs, device=self.device)
             self.obs = self.preprocess_obs(self.obs)
             self.ob_features = self.vision_head(self.obs)
             self.ob_latents = self.encoder(self.ob_features).latents
             self.w_pmasks.fill_(1)
             self.w_pmasks[:,-1] = 0
             self.w_obs.fill_(0)
             self.w_obs[:,-1] = self.obs
             self.w_ob_latents.fill_(0)
             self.w_ob_latents[:,-1] = self.ob_latents
             LOG('Envs reset')
             
             # Rollout buffers (used only during current rollout, no data forwarding)
             self.r_rewards = torch.zeros(self.envs_count, dataset.rollout_steps_count)
             self.r_dones = torch.zeros(self.envs_count, dataset.rollout_steps_count)
            
             self.is_reset = True
             self.levels_passed_counters.fill(0)
        
    def rollout(self, dataset, rollout_env_story_sampler=None, with_timing_counters=False, with_control_data=False):
        self.story_counters.clear()
        
        if not self.is_reset:
            self.reset(dataset, rollout_env_story_sampler=rollout_env_story_sampler)
            assert self.is_reset
            
        out_pmasks, out_obs, out_ob_latents, out_actions, out_action_log_probs, out_values, out_next_action_plan_logits = (
            dataset.pmasks,
            dataset.obs,
            dataset.ob_latents,
            dataset.actions,
            dataset.action_log_probs,
            dataset.values,
            dataset.next_action_plan_logits,
        )
        out_next_obs, out_next_ob_latents, out_dones, out_advantages, out_returns = (
            dataset.next_obs,
            dataset.next_ob_latents,
            dataset.dones,
            dataset.advantages,
            dataset.returns,
        )
        assert dataset.rollout_steps_count + dataset.prologue_size == out_pmasks.shape[1]
        
        # ROLLOUT
        assert torch.all(self.w_obs[:,-1] == self.obs)
        assert torch.all(self.w_ob_latents[:,-1] == self.ob_latents)

        if dataset.prologue_size > 0:
            # fill prologues with data from the prev rollout
            # For ob_latents prologue is mandatory since this window is reconstructed. For others - only to keep shape the same
            out_obs[:,:dataset.prologue_size] = out_obs[:,-dataset.prologue_size:]
            out_ob_latents[:,:dataset.prologue_size] = out_ob_latents[:,-dataset.prologue_size:]

        life_stats = []
        episode_stats = []
        control_data = defaultdict(list)
        timing_counters = defaultdict(list)

        def collect_stats(key, infos, stats):
            prefix = lu.when('life' in key, 'life', 'episode')
            
            assert len(infos[key]) == len(self.env_inds)
            
            for env_ind in np.argwhere(infos[key]).ravel():
                stats.append(dict(
                    env_ind=env_ind.item(),
                    r=infos[prefix + '_returns'][env_ind],
                    l=infos[prefix + '_lengths'][env_ind],
                    lp=self.levels_passed_counters[env_ind], # no difference between episode and life stats, yet
                ))

        with torch_utils.eval_mode(self.vision_head), torch_utils.eval_mode(self.encoder), torch_utils.eval_mode(self.agent), torch.no_grad():
            for rollout_step in range(dataset.rollout_steps_count):
                t0 = time.monotonic()
                
                out_index = dataset.prologue_size + rollout_step
                out_pmasks[:,out_index] = self.w_pmasks
                out_obs[:,out_index] = self.obs
                out_ob_latents[:,out_index] = self.ob_latents

                if with_control_data:
                    control_data['pmasks'].append(self.w_pmasks.cpu().numpy().copy())
                    control_data['obs'].append(self.w_obs.cpu().numpy().copy())
                    control_data['ob_latents'].append(self.w_ob_latents.cpu().numpy().copy())

                timing_counters['out_ob_latents'].append(time.monotonic() - t0)
                t0 = time.monotonic()

                # Agent thinks here
                agent_result = self.agent(
                    ob_latents=self.w_ob_latents,
                    padding_masks=self.w_pmasks,
                )

                out_actions[:,out_index] = agent_result.actions
                out_action_log_probs[:,out_index] = agent_result.action_log_probs
                out_values[:,out_index] = agent_result.values

                if rollout_step > 0: # population of out_next_action_plan_logits is also done during ADVANTAGES calc
                    assert (out_index - 1) >= 0
                    # Here we must patch PREVIOUS record!
                    # Note: if we just started from resetted env then we will patch record which belong to old env which is not quite correct.
                    # We workaround this during optimization by looking at dones flag - we won't take into account next_action_plan_logits for terminal records
                    out_next_action_plan_logits[:,out_index-1] = agent_result.action_plan_logits  

                if with_control_data:
                    control_data['actions'].extend(agent_result.actions.ravel().tolist())
                    control_data['action_log_probs'].extend(agent_result.action_log_probs.ravel().tolist())
                    control_data['action_plan_logits'].append(agent_result.action_plan_logits.cpu().numpy().copy())
                    control_data['values'].extend(agent_result.values.ravel().tolist())
                
                timing_counters['get_action_and_value'].append(time.monotonic() - t0)
                t0 = time.monotonic()
                
                # Interact with environments
                self.obs, rewards, terminations, truncations, infos = self.envs.step(agent_result.actions.cpu().numpy().ravel())
                self.obs = torch.tensor(self.obs, device=self.device)
                self.obs = self.preprocess_obs(self.obs)
                self.ob_features = self.vision_head(self.obs)
                self.ob_latents = self.encoder(self.ob_features).latents
                
                # Update sliding windows which are used to feed agent
                self.w_pmasks = self.w_pmasks.roll(shifts=-1, dims=1)
                self.w_pmasks[:,-1] = 0
                self.w_obs = self.w_obs.roll(shifts=-1, dims=1)
                self.w_obs[:,-1] = self.obs
                self.w_ob_latents = self.w_ob_latents.roll(shifts=-1, dims=1)
                self.w_ob_latents[:,-1] = self.ob_latents # here we store NEXT obs

                out_next_obs[:,out_index] = self.obs
                out_next_ob_latents[:,out_index] = self.ob_latents

                need_reset_envs = np.logical_or(terminations, truncations)
                dones = need_reset_envs.copy()

                if self.is_episodic_life and np.any(infos['is_life_lost']):
                    # Manually trigger done flag and truncate observations window (i.e. leave only current observation for new life)
                    # for envs which hit life loss
                    life_lost_envs = infos['is_life_lost']
                    # life_lost_envs must include need_reset_envs, i.e. it must be wider (superset)
                    assert np.all((need_reset_envs & life_lost_envs) == need_reset_envs), (need_reset_envs, life_lost_envs)
                    dones[life_lost_envs] = True

                if np.any(infos['is_level_passed']):
                    level_passed_envs = infos['is_level_passed']
                    self.levels_passed_counters[level_passed_envs] = self.levels_passed_counters[level_passed_envs] + 1
                
                self.r_dones[:,rollout_step] = torch.tensor(dones, dtype=torch.float)
                self.r_rewards[:,rollout_step] = torch.tensor(rewards, dtype=torch.float)

                if with_control_data:
                    control_data['dones'].extend(torch.tensor(dones, dtype=torch.float).tolist())
                
                collect_stats('is_life_lost', infos, life_stats)
                collect_stats('is_episode_over', infos, episode_stats)
                timing_counters['env.step'].append(time.monotonic() - t0)
        
                # Reset envs which hit an episode end (real one), for resetted envs get new observations
                if np.any(need_reset_envs):
                    t00 = time.monotonic()
                    reset_options = dict(reset_mask=need_reset_envs)
                    reset_options.update(self.get_rams_and_ram_patches(rollout_env_story_sampler, np.flatnonzero(need_reset_envs)))
                    reset_obs, _ = self.envs.reset(options=reset_options)
                    reset_obs = torch.tensor(reset_obs[need_reset_envs]).to(self.device)
                    reset_obs = self.preprocess_obs(reset_obs)
                    reset_ob_features = self.vision_head(reset_obs)
                    self.obs[need_reset_envs] = reset_obs
                    self.w_obs[:,-1] = self.obs
                    self.ob_latents[need_reset_envs] = self.encoder(reset_ob_features).latents
                    self.w_ob_latents[:,-1] = self.ob_latents
                    timing_counters['env.reset'].append(time.monotonic() - t00)

                if np.any(dones):
                    self.w_pmasks[dones] = 1
                    self.w_pmasks[dones,-1] = 0
                    self.levels_passed_counters[dones] = 0

            t0 = time.monotonic()
            
            # ADVANTAGES
            # Look ahead for one step
            agent_result = self.agent(
                ob_latents=self.w_ob_latents,
                padding_masks=self.w_pmasks,
            )
            out_next_action_plan_logits[:,-1] = agent_result.action_plan_logits 

            if with_control_data:
                control_data['lookahead_action_plan_logits'].append(agent_result.action_plan_logits.cpu().numpy().copy())
            
            rewards = self.r_rewards.to(self.device, non_blocking=True)
            dones = self.r_dones.to(self.device, non_blocking=True)
            values = out_values[:,-dataset.rollout_steps_count:]
            advantages = torch.zeros_like(out_advantages[:,-dataset.rollout_steps_count:])
            assert rewards.shape[1] == dataset.rollout_steps_count
            assert dones.shape[1] == dataset.rollout_steps_count
            assert values.shape[1] == dataset.rollout_steps_count
            assert advantages.shape[1] == dataset.rollout_steps_count
            lastgaelam = 0
            
            for t in reversed(range(dataset.rollout_steps_count)):
                if t == dataset.rollout_steps_count - 1:
                    nextnonterminal = 1.0 - dones[:,t]
                    nextnonterminal = nextnonterminal.to(self.device, non_blocking=True)
                    nextvalues = agent_result.values
                else:
                    nextnonterminal = 1.0 - dones[:,t]
                    nextvalues = values[:,t+1]
                
                delta = rewards[:,t] + HP.ppo.gamma * nextvalues * nextnonterminal - values[:,t]
                lastgaelam = delta + HP.ppo.gamma * HP.ppo.gae_lambda * nextnonterminal * lastgaelam
                advantages[:,t] = lastgaelam

            out_advantages[:,-dataset.rollout_steps_count:] = advantages
            returns = advantages + out_values[:,-dataset.rollout_steps_count:]
            out_returns[:,-dataset.rollout_steps_count:] = returns
            out_dones[:,-dataset.rollout_steps_count:] = dones
    
            if with_control_data:
                control_data['advantages'].extend(advantages.ravel().tolist())
                control_data['returns'].extend(returns.ravel().tolist())
                
            timing_counters['advantages'].append(time.monotonic() - t0)
    
            return dict(
                life_stats=life_stats,
                episode_stats=episode_stats,
                timing_counters=lu.when(with_timing_counters, timing_counters, None),
                control_data=lu.when(with_control_data, control_data, None),
                story_counters=self.story_counters,
            )

    def get_rams_and_ram_patches(self, rollout_env_story_sampler, env_inds):
        if not rollout_env_story_sampler:
            return {}
            
        rams = {}
        ram_patches = {}
        
        for env_ind in env_inds:
            story = rollout_env_story_sampler.sample()
            self.story_counters[story.desc] += 1
            # print(f'kms@ {int(env_ind)=}->{story.desc=}')

            if story.ram is not None:
                rams[int(env_ind)] = story.ram

            if story.ram_patch is not None:
                ram_patches[int(env_ind)] = self.env_ram_patcher(story.ram_patch)

        result = {}
        result.update(lu.when(rams, lambda: dict(rams=rams), {}))
        result.update(lu.when(ram_patches, lambda: dict(ram_patches=ram_patches), {}))
        return result

## Test

### rollout: performance

In [45]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = lu.when(CONFIG.is_cuda, 32, 8)
t.timing_counters = defaultdict(list)
t.rollout_steps_count = 128
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=RT.vision_head.params.ob_shape,
    d_model=RT.agent.params.d_model, 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=RT.agent.params.sequence_length, 
    action_plan_length=RT.agent.params.action_plan_length,
    actions_count=RT.agent.params.actions_count,
)
t.dataset.init_iteration(batches_count=4, is_shuffled=True)
t.rm = RolloutManager(
    vision_head=RT.vision_head,
    encoder=RT.encoder,
    agent=RT.agent, 
    envs_count=t.envs_count, 
    random_seed=HP.general.random_seed,
)

t.global_steps_count = t.rollout_steps_count * t.envs_count * 10
t.step = 0

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.rr = t.rm.rollout(t.dataset, with_timing_counters=True)

        for t.timing_counter_key, t.timing_counter_times in t.rr['timing_counters'].items():
            t.timing_counters[t.timing_counter_key].extend(t.timing_counter_times)

        step_inc = t.rollout_steps_count * t.envs_count
        t.step += step_inc
        pbar.update(step_inc)

list(map(lambda kv: (kv[0], np.array(kv[1]).mean().item()), t.timing_counters.items()))

  0%|          | 0/10240 [00:00<?, ?it/s]

Envs reset


[('out_ob_latents', 9.350596919830423e-05),
 ('get_action_and_value', 0.0085883969139104),
 ('env.step', 0.02088614376480109),
 ('advantages', 0.01618662364780903),
 ('env.reset', 0.02195232001280314)]

### rollout: pmasks, obs, ob_latents, etc.

In [46]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = lu.when(CONFIG.is_cuda, 32, 8)
t.df_columns = defaultdict(list)
t.sequence_length = RT.agent.params.sequence_length
t.rollout_steps_count = 128
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=RT.vision_head.params.ob_shape,
    d_model=RT.agent.params.d_model, 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=RT.agent.params.sequence_length, 
    action_plan_length=RT.agent.params.action_plan_length,
    actions_count=RT.agent.params.actions_count,
)
t.dataset.init_iteration(batches_count=4, is_shuffled=False)
t.rm = RolloutManager(
    vision_head=RT.vision_head,
    encoder=RT.encoder,
    agent=RT.agent, 
    envs_count=t.envs_count, 
    random_seed=HP.general.random_seed,
)

t.global_steps_count = t.rollout_steps_count * t.envs_count * 5
t.step = 0

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.stats_afs = dict(
            life_stats=dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter(), lp=RecursiveAverageFilter()),
            episode_stats=dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter(), lp=RecursiveAverageFilter()),
        )

        t.rr = t.rm.rollout(t.dataset, with_control_data=True)

        for t.k in t.stats_afs:
            for t.stats_item in t.rr[t.k]:
                t.stats_afs[t.k]['l'](t.stats_item['l'])
                t.stats_afs[t.k]['r'](t.stats_item['r'])
                t.stats_afs[t.k]['lp'](t.stats_item['lp'])

        t.control_pmasks = torch.tensor(np.array(t.rr['control_data']['pmasks'])).to(CONFIG.cuda_device)
        t.control_obs = torch.tensor(np.array(t.rr['control_data']['obs'])).to(CONFIG.cuda_device)
        t.control_ob_latents = torch.tensor(np.array(t.rr['control_data']['ob_latents'])).to(CONFIG.cuda_device)
        assert len(t.control_pmasks) == t.rollout_steps_count
        assert len(t.control_obs) == t.rollout_steps_count
        assert len(t.control_ob_latents) == t.rollout_steps_count

        # Verify that we get exactly the same windows (pmasks, ob_latents, actions) as during rollout
        # Also make sure important conditions are kept
        for t.batch in t.dataset:
            t.b_orig_w_pmasks = t.control_pmasks[t.batch.step_inds, t.batch.env_inds]
            assert torch.all(t.b_orig_w_pmasks == t.batch.pmasks)
            # current items are LAST items and are always unmasked, make sure pmasks reflects this fact
            assert torch.all(t.batch.pmasks[:,-1] == 0) 
            # make sure there are no pmasks with all ones
            t.pmasks_counts = t.batch.pmasks.sum(axis=1)
            assert torch.all(t.pmasks_counts < t.sequence_length)

            # make sure that zeros (unmasking) are pushed to pmasks right to left
            for t.l in range(1, t.sequence_length): # 1,2,..,sequence_length-1
                t.expected_pmask = torch.zeros(t.sequence_length, dtype=t.batch.pmasks.dtype).to(t.batch.pmasks.device)
                t.expected_pmask[:t.l] = 1
                t.seq_len_mask = (t.pmasks_counts == t.l)
                assert torch.all(t.batch.pmasks[t.seq_len_mask] == t.expected_pmask)

            t.b_orig_w_obs = t.control_obs[t.batch.step_inds, t.batch.env_inds]
            assert torch.all(t.b_orig_w_obs == t.batch.obs)
            
            t.b_orig_w_ob_latents = t.control_ob_latents[t.batch.step_inds, t.batch.env_inds]
            assert torch.all(t.b_orig_w_ob_latents == t.batch.ob_latents)

            # Reenable in case dones will be a windowed object
            if False:
                t.b_orig_w_dones = t.control_dones[t.batch.step_inds, t.batch.env_inds]
                assert torch.all(t.b_orig_w_dones == t.batch.dones)

                # Since dones is a true windowed object (as opposed to ob_latents which is a reconstructed one) 
                # then we demand that it doesn't contain leftovers from prev runs (episodes). 
                # As such if current item (last item in sequence) is done then all prev dones in this window must be 0
                t.dones_counts = t.batch.dones.sum(axis=1)
                assert torch.all(t.dones_counts <= 1)
                t.any_dones_mask = (t.dones_counts > 0)
                assert torch.all(t.batch.dones[t.any_dones_mask,-1] == 1)
                assert torch.all(t.batch.dones[t.any_dones_mask,:-1] == 0)
                # For any done env all items must be present (pmask)
                assert torch.all(t.batch.pmasks[t.any_dones_mask,:-1] == 0)
        
        # Verify other data via control data
        actions = []
        action_log_probs = []
        values = []
        dones = []
        advantages = []
        returns = []
        
        for t.batch in t.dataset:
            actions.extend(t.batch.actions.cpu().numpy())
            action_log_probs.extend(t.batch.action_log_probs.cpu().numpy())
            values.extend(t.batch.values.cpu().numpy())
            dones.extend(t.batch.dones.cpu().numpy())
            advantages.extend(t.batch.advantages.cpu().numpy())
            returns.extend(t.batch.returns.cpu().numpy())

        assert np.all(np.sort(np.array(actions)) == np.sort(t.rr['control_data']['actions']))
        assert np.all(np.sort(np.array(action_log_probs)) == np.sort(t.rr['control_data']['action_log_probs']))
        assert np.all(np.sort(np.array(values)) == np.sort(t.rr['control_data']['values']))
        assert np.all(np.sort(np.array(dones)) == np.sort(t.rr['control_data']['dones']))
        assert np.all(np.sort(np.array(advantages)) == np.sort(t.rr['control_data']['advantages']))
        assert np.all(np.sort(np.array(returns)) == np.sort(t.rr['control_data']['returns']))
            
        step_inc = t.rollout_steps_count * t.envs_count
        t.step += step_inc
        pbar.update(step_inc)

        t.df_columns['life_stats.n'].append(t.stats_afs['life_stats']['l'].n)
        t.df_columns['life_stats.r'].append(t.stats_afs['life_stats']['r'].v)
        t.df_columns['life_stats.l'].append(t.stats_afs['life_stats']['l'].v)
        t.df_columns['life_stats.lp'].append(t.stats_afs['life_stats']['lp'].v)
        t.df_columns['episode_stats.n'].append(t.stats_afs['episode_stats']['l'].n)
        t.df_columns['episode_stats.r'].append(t.stats_afs['episode_stats']['r'].v)
        t.df_columns['episode_stats.l'].append(t.stats_afs['episode_stats']['l'].v)
        t.df_columns['episode_stats.lp'].append(t.stats_afs['episode_stats']['lp'].v)

pd.DataFrame(t.df_columns).style.format("{:.2f}")

  0%|          | 0/5120 [00:00<?, ?it/s]

Envs reset


,life_stats.n,life_stats.r,life_stats.l,life_stats.lp,episode_stats.n,episode_stats.r,episode_stats.l,episode_stats.lp
0,4.00,10.00,385.00,0.00,0.00,0.00,0.00,0.00
1,10.00,15.00,459.20,0.00,0.00,0.00,0.00,0.00
2,12.00,6.67,388.83,0.00,3.00,20.00,1281.67,0.00
3,10.00,13.00,411.00,0.00,3.00,63.33,1767.67,0.00
4,10.00,17.00,414.40,0.00,3.00,80.00,2053.67,0.00


### rollout: next_obs, next_ob_latents, next_action_plan_logits

In [47]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 1
t.rollout_steps_count = 256
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=RT.vision_head.params.ob_shape,
    d_model=RT.agent.params.d_model, 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=RT.agent.params.sequence_length, 
    action_plan_length=RT.agent.params.action_plan_length,
    actions_count=RT.agent.params.actions_count,
)
t.dataset.init_iteration(batches_count=1, is_shuffled=False)
t.rm = RolloutManager(
    vision_head=RT.vision_head,
    encoder=RT.encoder,
    agent=RT.agent, 
    envs_count=t.envs_count, 
    random_seed=HP.general.random_seed,
)
    
t.global_steps_count = t.rollout_steps_count * 4
t.step = 0
t.any_dones = False

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.rr = t.rm.rollout(t.dataset, with_control_data=True)

        t.batch = next(iter(t.dataset)) # one batch only
        
        # Internal check: verify that within window items which are not terminal and which are present (pmasks) have matching next obs
        # Disabled because one need to return is_ptf for Dataset
        # t.bw_inds = t.dataset.bw_inds_stem + t.batch.b_inds.unsqueeze(1)
        # t.bw_inds = t.bw_inds.view(-1)
        # t.bw_dones = t.dones[t.bw_inds].view(t.batch.obs.shape[:2])

        # for t.i in range(t.bw_dones.shape[0]):
        #     for t.j in range(t.bw_dones.shape[1] - 1):
        #         if t.bw_dones[t.i,t.j] == 0 and t.batch.obs_pmasks[t.i,t.j] == 0:
        #             assert torch.all(t.batch.obs[t.i,t.j+1] == t.batch.next_obs[t.i,t.j])

        # External check: verify that all running (non terminal) items must have next item where obs == next_obs
        t.any_dones = t.any_dones or torch.any(t.batch.dones.bool())
        t.running = ~t.batch.dones.bool()
        assert len(t.running) == len(t.batch.obs)
        
        for i, is_running in enumerate(t.running):
            if is_running:
                if (i + 1) < len(t.batch.obs):
                    assert torch.all(t.batch.next_obs[i] == t.batch.obs[i+1,-1])
                else:
                    # Don't have access to next obs since this is a last item in rollout batch
                    pass
            else:
                # We've done env, t.batch.obs[i+1] will refer to a different (resetted) env, so can't compare with the latter
                # t.batch.next_obs could be of any data - it will be ignored anyway
                pass 
        
        # External check: verify that all running (non terminal) items must have next item where ob_latents == next_ob_latents
        t.any_dones = t.any_dones or torch.any(t.batch.dones.bool())
        t.running = ~t.batch.dones.bool()
        assert len(t.running) == len(t.batch.ob_latents)
        
        for i, is_running in enumerate(t.running):
            if is_running:
                if (i + 1) < len(t.batch.ob_latents):
                    assert torch.all(t.batch.next_ob_latents[i] == t.batch.ob_latents[i+1,-1])
                else:
                    # Don't have access to next ob_latents since this is a last item in rollout batch
                    pass
            else:
                # We've done env, t.batch.ob_latents[i+1] will refer to a different (resetted) env, so can't compare with the latter
                # t.batch.next_ob_latents could be of any data - it will be ignored anyway
                pass 
            
        t.control_action_plan_logits = torch.tensor(np.array(t.rr['control_data']['action_plan_logits'])).to(CONFIG.cuda_device)
        t.control_lookahead_action_plan_logits = torch.tensor(np.array(t.rr['control_data']['lookahead_action_plan_logits'])).to(CONFIG.cuda_device)
        assert len(t.control_action_plan_logits) == t.rollout_steps_count
        assert len(t.running) == len(t.batch.next_action_plan_logits)
        
        for i, is_running in enumerate(t.running):
            if is_running:
                if (i + 1) < len(t.batch.next_action_plan_logits):
                    assert torch.all(t.batch.next_action_plan_logits[i] == t.control_action_plan_logits[i+1])
                else:
                    assert torch.all(t.batch.next_action_plan_logits[i] == t.control_lookahead_action_plan_logits)
            else:
                # We've done env, t.batch.next_action_plan_logits could of any data - it will be ignored anyway
                pass 
            
        step_inc = t.rollout_steps_count
        t.step += step_inc
        pbar.update(step_inc)

assert t.any_dones

  0%|          | 0/1024 [00:00<?, ?it/s]

Envs reset


# Test

## TestManager

In [51]:
class TestManager:
    def __init__(self, vision_head, encoder, agent, envs_count, rams=None, ram_patches=None, device=None):
        self.preprocess_obs = ob_preprocessor.ObPreprocessor(vision_head.params.ob_shape, cast_to_float=False)
        self.vision_head = vision_head
        self.encoder = encoder
        self.agent = agent
        self.envs_count = envs_count
        self.device = lu.coalesce(device, CONFIG.cuda_device)
       
        self.envs = create_envs(envs_count, is_strict_disabled_autoreset=False)
        LOG(f'Envs created: {envs_count} envs', when=not CONFIG.is_interactive)

        rams = lu.when(rams is None or len(rams) == 0, {None: None}, rams)
        ram_patches = lu.when(ram_patches is None or len(ram_patches) == 0, [None], ram_patches)
        assert len(rams) > 0
        assert len(ram_patches) > 0
        self.stories = []

        for ram_id, ram in rams.items():
            for ram_patch in ram_patches:
                self.stories.append((ram_id, ram, ram_patch))

        assert self.stories
        
        self.env_ram_patcher = EnvRamPatcher()

    def play(self, max_steps_count, random_seed, break_on_level_passed=False, is_episodic_life=True):
        sequence_length = self.agent.params.sequence_length
        w_pmasks = torch.ones(self.envs_count, sequence_length).to(self.device)
        w_ob_latents = torch.zeros(self.envs_count, sequence_length, self.agent.params.d_model).to(self.device)
        stats = defaultdict(lambda: np.zeros((self.envs_count, len(self.stories))))

        def accumulate_stats(env_ind, story_ind, infos):
            stats['returns'][env_ind,story_ind] = infos['episode_returns'][env_ind]
            stats['lengths'][env_ind,story_ind] = infos['episode_lengths'][env_ind]
            stats['levels_passed'][env_ind,story_ind] = infos['levels_passed'][env_ind]
            
        with torch_utils.eval_mode(self.vision_head), torch_utils.eval_mode(self.encoder), torch_utils.eval_mode(self.agent), torch.no_grad():
            for story_ind, (ram_id, ram, ram_patch) in enumerate(self.stories):
                story_desc = f'{ram_id}/'
                story_desc += lu.when(ram_patch, lambda: '+'.join(ram_patch), 'None')
                LOG(f'Story for test play="{story_desc}"', when=not CONFIG.is_interactive)

                if ram_patch is not None:
                    ram_patch = self.env_ram_patcher(ram_patch)
    
                assert ram_patch is None or isinstance(ram_patch, dict)
                
                running_envs = set(range(self.envs_count))
                reset_options = {}
                reset_options.update(lu.when(ram is not None, lambda: dict(rams={-1: ram}), {}))
                reset_options.update(lu.when(ram_patch is not None, lambda: dict(ram_patches={-1: ram_patch}), {}))
                obs, _ = self.envs.reset(seed=random_seed, options=reset_options)
                obs = torch.tensor(obs).to(self.device)
                obs = self.preprocess_obs(obs)
                features = self.vision_head(obs)
                w_pmasks.fill_(1)
                w_pmasks[:,-1] = 0
                w_ob_latents.fill_(0)
                w_ob_latents[:,-1] = self.encoder(features).latents
                t0 = time.monotonic()

                for step in range(max_steps_count): 
                    if step > 0 and step % 100 == 0:
                        LOG(f'Made 100 steps in {time.monotonic() - t0:.3f}s, total steps made {step}/{max_steps_count}', when=not CONFIG.is_interactive)
                        t0 = time.monotonic()

                    agent_result = self.agent(
                        ob_latents=w_ob_latents,
                        padding_masks=w_pmasks,
                    )

                    np_actions = agent_result.actions.cpu().numpy()
                    obs, rewards, terminations, truncations, infos = self.envs.step(np_actions) # for finished envs calling envs is just a dry run
                    obs = torch.tensor(obs).to(self.device)
                    obs = self.preprocess_obs(obs) 
                    features = self.vision_head(obs)
                    
                    w_pmasks = w_pmasks.roll(shifts=-1, dims=1)
                    w_pmasks[:,-1] = 0
                    w_ob_latents = w_ob_latents.roll(shifts=-1, dims=1)
                    w_ob_latents[:,-1] = self.encoder(features).latents
            
                    if is_episodic_life and np.any(infos['is_life_lost']):
                        life_lost_envs = infos['is_life_lost']
                        w_pmasks[life_lost_envs,:-1] = 1 # mask all but the last
                        w_ob_latents[life_lost_envs,:-1] = 0 # wipe out all but the last

                    if break_on_level_passed and np.any(infos['is_level_passed']):
                        for env_ind in np.argwhere(infos['is_level_passed']).ravel():
                            env_ind = env_ind.item()
                            
                            if env_ind in running_envs:
                                accumulate_stats(env_ind, story_ind, infos)
                                running_envs.remove(env_ind)

                    done_envs = np.logical_or(terminations, truncations)

                    if np.any(done_envs):
                        for env_ind in np.argwhere(done_envs).ravel():
                            env_ind = env_ind.item()
                            
                            if env_ind in running_envs:
                                accumulate_stats(env_ind, story_ind, infos)
                                running_envs.remove(env_ind)

                    if not running_envs:
                        break

                for env_ind in running_envs: # this is executed when we hit max_steps_count but some envs are not done yet
                    accumulate_stats(env_ind, story_ind, infos)

        return stats

## Test

In [52]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 4
t.max_steps_count = 100
t.test_manager = TestManager(
    vision_head=RT.vision_head,
    encoder=RT.encoder,
    agent=RT.agent, 
    envs_count=t.envs_count, 
    ram_patches=[['no_score', 'full_igloo'], None],
)
t.test_manager.play(
    max_steps_count=t.max_steps_count, 
    random_seed=42,
    break_on_level_passed=True,
)

defaultdict(<function __main__.TestManager.play.<locals>.<lambda>()>,
            {'returns': array([[10., 10.],
                    [20., 20.],
                    [10., 30.],
                    [20., 10.]]),
             'lengths': array([[411., 411.],
                    [403., 403.],
                    [425., 425.],
                    [429., 429.]]),
             'levels_passed': array([[0., 0.],
                    [0., 0.],
                    [0., 0.],
                    [0., 0.]])})

## Configure

In [53]:
# @launchit.disable
# @launchit.collect
# Test params
HP.test.env_rams = [
    'com.develorium.neurolab.frostbite_ram:level5_101:1',
]
HP.test.env_ram_patches = None
HP.test.break_on_level_passed = False
# @launchit.stop

## Initrd

In [54]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig' and lu.coalesce(HP.test.env_rams, []):
    for ram_id in HP.test.env_rams:
        coords = hp_parse_artifact_source(ram_id)
        CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='pkl', asset_classifier=coords.asset_classifier, maven_group_id=coords.group_id)
# @launchit.stop

## Create

In [56]:
test_env_rams = None

if lu.coalesce(HP.test.env_rams, []):
    test_env_rams = {}
    
    for ram_id in HP.test.env_rams:
        coords = hp_parse_artifact_source(ram_id)
        ram_asset = RT.artifact_registry.get_asset_content(
            coords.model_name, 
            coords.model_version, 
            asset_ext='pkl', 
            asset_classifier=coords.asset_classifier, 
            maven_group_id=coords.group_id,
        )
        
        with io.BytesIO(ram_asset) as b:
            ram = pickle.load(b)
            test_env_rams[ram_id] = ram

        LOG(f'Test env RAM "{ram_id}" loaded')

    LOG(f'{len(test_env_rams)=}')

RT.test_manager = TestManager(
    vision_head=RT.vision_head,
    encoder=RT.encoder,
    agent=RT.agent, 
    envs_count=HP.env.count, 
    rams=test_env_rams,
    ram_patches=HP.test.env_ram_patches,
)
LOG('TestManager created')

Test env RAM "com.develorium.neurolab.frostbite_ram:level5_101:1" loaded
len(test_env_rams)=1


AttributeError: 'Runtime' object has no attribute 'test_manager'

# TRAIN

## EdgeDetector

In [51]:
# @launchit.collect_explore
# https://share.google/aimode/910bVkcGs9nvCh9Q8
class EdgeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        # Sobel filters to extract horizontal and vertical edges
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        # Register as buffers so they move to the correct GPU automatically
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    ForwardResult = namedtuple('ForwardResult', 'grad_x, grad_y, magnitude')    
    
    def forward(self, x):
        # x shape: (Batch, 1, H, W)
        padded = F.pad(x, (1, 1, 1, 1), mode='replicate') # Pad edges to keep sizes matching
        grad_x = F.conv2d(padded, self.sobel_x)
        grad_y = F.conv2d(padded, self.sobel_y)
        magnitude = torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-6)
        return EdgeDetector.ForwardResult(grad_x=grad_x, grad_y=grad_y, magnitude=magnitude)

## Configure

In [52]:
# @launchit.disable
# @launchit.collect
# Training procedure params (PPO related) 
HP.ppo.global_steps_count = 10_000 # total number of steps 
HP.ppo.rollout_steps_count = 128 # how many steps to run in a single policy rolllout
HP.ppo.rollout_env_rams = [
    'com.develorium.neurolab.frostbite_ram:level1_101:1',
]
HP.ppo.rollout_env_ram_patches = [
    ['last_life', 'full_igloo', 'bailey_right_at_the_igloo_door', 'temperature_10'],
    ['last_life', 'full_igloo', 'bailey_very_near_igloo_door', 'temperature_10'],  
    ['last_life', 'full_igloo', 'bailey_near_center', 'temperature_10'],
    ['last_life', 'one_remaining_igloo', 'bailey_near_center', 'temperature_10'],
    ['last_life', 'three_remaining_igloo', 'bailey_near_center', 'temperature_20'],
    ['last_life', 'half_igloo', 'bailey_near_center'],
    ['last_life', 'half_igloo'],
    ['last_life'],
]
HP.ppo.rollout_env_stories = ['*;*']

HP.ppo.epochs_count = 2 
HP.ppo.batch_size = 128
HP.ppo.learn_rate = 'const(0.00025)'
HP.ppo.optimizer = 'AdamW'

HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
HP.ppo.ent_coef = 'const(0.05)' # coefficient of the entropy member within loss function
HP.ppo.consistency_coef = 0.1 # coefficient of the loss for action plan consistency between adjacent steps
HP.ppo.prediction_coef = 0.1 # coefficient of the loss for observation emb. prediction, if set to 0.0 the prediction loss is not used

HP.ppo.gamma = 0.997 # return discount factor gamma
HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
HP.ppo.target_kl = None # e target KL divergence threshold
HP.ppo.norm_adv = True # Toggles advantages normalization
# @launchit.stop

In [53]:
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': 'train',
 'launch_id': 0,
 'general': {'comment': None,
             'random_seed': 99,
             'is_torch_deterministic': True,
             'is_torch_compile': False,
             'is_torch_amp': True},
 'env': {'count': 32, 'is_episodic_life': True, 'actions_count': 6},
 'vision_head': {'parent': {'model': '18d_world_model_09:40',
                            'weights': '18d_world_model_09:40'},
                 'is_trainable': False},
 'encoder': {'parent': {'model': '18d_world_model_09:40',
                        'weights': '18d_world_model_09:40'},
             'is_trainable': True},
 'agent': {'parent': None,
           'sequence_length': 10,
           'action_plan_length': 10,
           'd_model': 256,
           'transformer': {'layers_count': 3,
                           'heads_count': 4,
                           'attention_backend': ['EFFICIENT_ATTENTION',
                                                 'MATH']},
           'is_trainable': True},
 '

## Initrd

In [54]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig' and lu.coalesce(HP.ppo.rollout_env_rams, []):
    for ram_id in HP.ppo.rollout_env_rams:
        coords = hp_parse_artifact_source(ram_id)
        CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='pkl', asset_classifier=coords.asset_classifier, maven_group_id=coords.group_id)
# @launchit.stop

## Create

In [55]:
ump = hp_parse_universal_module(HP.ppo.optimizer)
assert not ump.args
all_params = [p for p in RT.vision_head.parameters()] + [p for p in RT.encoder.parameters()] + [p for p in RT.agent.parameters()]
optimized_params = list(filter(lambda p: p.requires_grad, all_params))
optimizer = getattr(torch.optim, ump.module_name)(optimized_params, **ump.kwargs)
all_params_count = sum(p.numel() for p in all_params)
optimized_params_count = sum(p.numel() for p in optimized_params)
LOG(f'{all_params_count=:_}, {optimized_params_count=:_}')

ump = hp_parse_universal_module(HP.ppo.learn_rate)
lr_anneal = torch_utils.get_anneal(ump.module_name, *ump.args, **ump.kwargs)

ump = hp_parse_universal_module(HP.ppo.ent_coef)
ent_coef_anneal = torch_utils.get_anneal(ump.module_name, *ump.args, **ump.kwargs)

all_params_count=5_083_015, optimized_params_count=4_931_975


In [56]:
life_stats_afs = dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter(), lp=RecursiveAverageFilter())
episode_stats_afs = dict(l=RecursiveAverageFilter(), r=RecursiveAverageFilter(), lp=RecursiveAverageFilter())
clip_fracs_af = RecursiveAverageFilter()

dataset = Dataset(
    envs_count=HP.env.count,
    ob_shape=RT.vision_head.params.ob_shape,
    d_model=RT.agent.params.d_model, 
    rollout_steps_count=HP.ppo.rollout_steps_count, 
    sequence_length=RT.agent.params.sequence_length, 
    action_plan_length=RT.agent.params.action_plan_length,
    actions_count=RT.agent.params.actions_count,
)
dataset.init_iteration(batch_size=HP.ppo.batch_size, is_shuffled=True)

for name, v in dataset.main_data.items():
    if v is not None:
        LOG(f'{name:>28}: {str(v.device):>6}, {str(v.dtype):>15}, {v.shape}')

                      pmasks: cuda:0,   torch.float32, torch.Size([32, 137, 10])
                         obs: cuda:0,     torch.uint8, torch.Size([32, 137, 3, 89, 76])
                  ob_latents: cuda:0,   torch.float32, torch.Size([32, 137, 256])
                     actions: cuda:0,     torch.int64, torch.Size([32, 137])
            action_log_probs: cuda:0,   torch.float32, torch.Size([32, 137])
                      values: cuda:0,   torch.float32, torch.Size([32, 137])
     next_action_plan_logits: cuda:0,   torch.float32, torch.Size([32, 137, 10, 6])
                    next_obs: cuda:0,     torch.uint8, torch.Size([32, 137, 3, 89, 76])
             next_ob_latents: cuda:0,   torch.float32, torch.Size([32, 137, 256])
                       dones: cuda:0,   torch.float32, torch.Size([32, 137])
                  advantages: cuda:0,   torch.float32, torch.Size([32, 137])
                     returns: cuda:0,   torch.float32, torch.Size([32, 137])
                    env_inds: cud

In [57]:
rollout_env_rams = None

if lu.coalesce(HP.ppo.rollout_env_rams, []):
    rollout_env_rams = {}
    
    for ram_id in HP.ppo.rollout_env_rams:
        coords = hp_parse_artifact_source(ram_id)
        ram_asset = RT.artifact_registry.get_asset_content(
            coords.model_name, 
            coords.model_version, 
            asset_ext='pkl', 
            asset_classifier=coords.asset_classifier, 
            maven_group_id=coords.group_id,
        )
        
        with io.BytesIO(ram_asset) as b:
            ram = pickle.load(b)
            rollout_env_rams[ram_id] = ram

        LOG(f'Rollout env RAM "{ram_id}" loaded')

    LOG(f'{len(rollout_env_rams)=}')

Rollout env RAM "com.develorium.neurolab.frostbite_ram:level1_101:1" loaded
len(rollout_env_rams)=1


In [58]:
rollout_env_story_sampler = RolloutEnvStorySampler(rollout_env_rams, HP.ppo.rollout_env_ram_patches, HP.ppo.rollout_env_stories)
rollout_manager = RolloutManager(
    RT.vision_head,
    RT.encoder, 
    RT.agent, 
    envs_count=HP.env.count, 
    random_seed=HP.general.random_seed,
)

Envs created: 32 envs, thread_pool_size=None, thread_affinity_offset=None


In [59]:
edge_detector = EdgeDetector()
edge_detector = edge_detector.to(CONFIG.cuda_device)

In [ ]:
def create_is_follow_up(period=None):
    assert period is not None
    next_gate = 0
    
    def every_thunk(counter, is_last_step):
        nonlocal next_gate

        if counter >= next_gate:
            next_gate += period
            return True

        return is_last_step

    return every_thunk

# Make follow up to 10 times
is_follow_up = create_is_follow_up(HP.ppo.global_steps_count // 10)

In [ ]:
layout = {
    'test': {
        'levels_passed': ['Margin', [ 'test/levels_passed_mean', 'test/levels_passed_min', 'test/levels_passed_max']],
        'returns': ['Margin', [ 'test/returns_mean', 'test/returns_min', 'test/returns_max']],
        'lengths': ['Margin', [ 'test/lengths_mean', 'test/lengths_min', 'test/lengths_max']],
    }
}
RT.summary_writer.add_custom_scalars(layout)

## Train

In [61]:
metrics_suite = defaultdict(list)
timings = {}
global_step_inc = dataset.items_count
ceiled_global_steps_count = math.ceil(HP.ppo.global_steps_count / global_step_inc) * global_step_inc
is_action_plan_consistency_loss = RT.agent.params.action_plan_length > 1 and HP.ppo.consistency_coef > 0
is_prediction_loss = HP.ppo.prediction_coef > 0

for global_step in tqdm(range(0, ceiled_global_steps_count, global_step_inc), disable=not CONFIG.is_interactive):
    timings.clear()
    t0 = time.monotonic()
    
    # EXECUTE EXTERNAL COMMANDS
    if command := RT.command_listener.get_command():
        LOG(f'Got {command=}')
        
        if command == 'stop':
            RT.should_save_model = True
            break
        elif command == 'abort':
            RT.should_save_model = False
            break
        else:
            LOG(f'Ignoring unknown {command=}')
    
    # ANNEAL
    global_step_t = global_step / ceiled_global_steps_count
    lr = lr_anneal(global_step_t)
    for param_group in optimizer.param_groups: param_group["lr"] = lr
    ent_coef = ent_coef_anneal(global_step_t)

    # ROLLOUT
    t1 = time.monotonic()
    rr = rollout_manager.rollout(dataset, rollout_env_story_sampler=rollout_env_story_sampler)

    for stats_item_name, afs in zip(('life_stats', 'episode_stats'), (life_stats_afs, episode_stats_afs)):
        for stats_item in rr.get(stats_item_name, []):
            afs['l'](stats_item['l'])
            afs['r'](stats_item['r'])
            afs['lp'](stats_item['lp'])

    total_stories_used = sum(rr.get('story_counters', {}).values())
    LOG(f'Rollout done, {total_stories_used} stories used:', when=not CONFIG.is_interactive)

    for story_name, story_counter in rr.get('story_counters', {}).items():
        LOG(f'\t{story_counter:3} ({story_counter / total_stories_used * 100:3.2f}%): {story_name}', when=not CONFIG.is_interactive)
    
    timings['rollout'] = time.monotonic() - t1
    
    # OPTIMIZATION
    t1 = time.monotonic()
    assert not RT.vision_head.is_trainable
    if RT.vision_head.is_trainable: RT.vision_head.train()
    if RT.encoder.is_trainable: RT.encoder.train()
    if RT.agent.is_trainable: RT.agent.train()
    
    for epoch in range(HP.ppo.epochs_count):
        for batch_ind, batch in enumerate(dataset):
            batch_size = len(batch.obs)
            assert torch.all(batch.pmasks[:,-1] == 0)

            with torch.amp.autocast(device_type=CONFIG.cuda_device, dtype=torch.bfloat16, enabled=HP.general.is_torch_amp):
                if RT.encoder.is_trainable:
                    batch = batch._replace(
                        ob_latents=RT.encoder(RT.vision_head(batch.obs)).latents,
                        next_ob_latents=RT.encoder(RT.vision_head(batch.next_obs)).latents.detach(), # detach() is a must!!!
                    )
                
                agent_result = RT.agent(
                    padding_masks=batch.pmasks,
                    ob_latents=batch.ob_latents, 
                    actions=batch.actions, 
                    predict=is_prediction_loss,
                )
    
                new_action_log_probs = agent_result.action_log_probs
                new_action_plan_logits = agent_result.action_plan_logits
                new_action_entropies = agent_result.action_entropies
                new_values = agent_result.values
                b_action_log_probs = batch.action_log_probs
                b_values = batch.values
                b_next_action_plan_logits = batch.next_action_plan_logits
                b_dones = batch.dones
                b_advantages = batch.advantages
                b_returns = batch.returns
    
                # Policy loss
                if HP.ppo.norm_adv:
                    b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)
    
                logratio = new_action_log_probs - b_action_log_probs
                ratio = torch.exp(logratio)
                pgloss1 = -b_advantages * ratio
                pgloss2 = -b_advantages * torch.clamp(ratio, 1.0 - HP.ppo.clip_coef, 1.0 + HP.ppo.clip_coef)
                pg_loss = torch.max(pgloss1, pgloss2).mean()
    
                # Value loss
                v_loss_unclipped = (new_values - b_returns) ** 2 # (batch, )
                
                if HP.ppo.clip_vloss:
                    # https://share.google/aimode/Aui6vM2UqzODZvO4n
                    clipped_new_values = b_values + (new_values - b_values).clamp(min=-HP.ppo.clip_coef, max=HP.ppo.clip_coef)
                    v_loss_clipped = (clipped_new_values - b_returns) ** 2 # (batch, )
                    v_loss = torch.max(v_loss_unclipped, v_loss_clipped).mean()
                else:
                    v_loss = v_loss_unclipped.mean()
    
                # Action plan consistency loss
                action_plan_consistency_loss = 0 
                
                if is_action_plan_consistency_loss:
                    # kms@ CrossEntropy instead of MSE?
                    # Take action plans of running only records and compare to next - demand that they should not drift far from each other
                    b_running = ~b_dones.bool()
                    action_plan_consistency_loss = F.mse_loss(new_action_plan_logits[b_running,1:], b_next_action_plan_logits[b_running,:-1])
    
                # Entropy loss
                entropy_loss = new_action_entropies.mean()
    
                # Prediction loss of next observation
                prediction_loss = 0
                
                if is_prediction_loss:
                    pred_ob_latents = agent_result.pred_ob_latents
                    assert pred_ob_latents.shape == (batch_size, RT.agent.params.d_model)

                    true_ob_latents = batch.next_ob_latents
                    assert true_ob_latents.shape == (batch_size, RT.agent.params.d_model)
                    assert batch.next_ob_latents.requires_grad == False

                    norm_pred_ob_latents = F.normalize(pred_ob_latents, dim=-1)
                    norm_true_ob_latents = F.normalize(true_ob_latents, dim=-1)
                        
                    prediction_loss = F.mse_loss(norm_pred_ob_latents, norm_true_ob_latents)
                
            if (epoch == HP.ppo.epochs_count - 1) and (batch_ind == dataset.batches_count - 1):
                # measure contribution of each loss to total gradient
                # https://share.google/aimode/fETIzeu7FwUbO0tEr
                def get_total_norm():
                    grads = [p.grad for p in optimized_params if p.grad is not None]
                    return torch.nn.utils.get_total_norm(grads).item()
                
                optimizer.zero_grad()
                pg_loss.backward(retain_graph=True)
                RT.summary_writer.add_scalar(f'grad/a) total_by_policy_loss', get_total_norm(), global_step)
                
                optimizer.zero_grad() 
                (HP.ppo.vf_coef * v_loss).backward(retain_graph=True)
                RT.summary_writer.add_scalar(f'grad/a) total_by_value_loss', get_total_norm(), global_step)

                optimizer.zero_grad() 
                (HP.ppo.consistency_coef * action_plan_consistency_loss).backward(retain_graph=True)
                RT.summary_writer.add_scalar(f'grad/a) total_by_action_plan_consistency_loss', get_total_norm(), global_step)

                optimizer.zero_grad() 
                (HP.ppo.prediction_coef * prediction_loss).backward(retain_graph=True)
                RT.summary_writer.add_scalar(f'grad/a) total_by_prediction_loss', get_total_norm(), global_step)

                optimizer.zero_grad() 
                (ent_coef * entropy_loss).backward(retain_graph=True)
                RT.summary_writer.add_scalar(f'grad/a) total_by_entropy_loss', get_total_norm(), global_step)

            # Combined losses
            loss = (
                pg_loss 
                + (HP.ppo.vf_coef * v_loss)
                + (HP.ppo.consistency_coef * action_plan_consistency_loss)
                + (HP.ppo.prediction_coef * prediction_loss)
                - (ent_coef * entropy_loss)
            )
                
            optimizer.zero_grad()
            loss.backward()
            group_grad_norms = torch_utils.calc_group_grad_norms(RT.agent)

            if RT.encoder.is_trainable:
                group_grad_norms.update(torch_utils.calc_group_grad_norms(RT.encoder))
            
            grad_norm = torch.nn.utils.clip_grad_norm_(optimized_params, max_norm=HP.ppo.max_grad_norm)
            optimizer.step()

            with torch.no_grad():
                clip_fracs_af(((ratio - 1.0).abs() > HP.ppo.clip_coef).float().mean().item())

    LOG(f'Optimization done', when=not CONFIG.is_interactive)
    timings['optimization'] = time.monotonic() - t1

    # FOLLOW-UP
    if is_follow_up(global_step, is_last_step=global_step+global_step_inc>=HP.ppo.global_steps_count):
        t1 = time.monotonic()
        test_play_stats = RT.test_manager.play(
            max_steps_count=5000, 
            random_seed=HP.general.random_seed+HP.env.count+1,
            break_on_level_passed=HP.test.break_on_level_passed,
        )

        for metric_name in test_play_stats:
            metric_matrix = test_play_stats[metric_name]
            log_string = []

            for reduce_method in ('mean', 'min', 'max'):
                metric_value = getattr(metric_matrix, reduce_method)().item()
                metrics_suite[f'test/{metric_name}_{reduce_method}'].append(metric_value)
                RT.summary_writer.add_scalar(f'test/{metric_name}_{reduce_method}', metric_value, global_step)
                log_string.append(f'{reduce_method}={metric_value}')
            
            LOG(f'{global_step=}, test/{metric_name}: {', '.join(log_string)}', when=not CONFIG.is_interactive)

        timings['test'] = time.monotonic() - t1

    # REPORT
    t_end = time.monotonic()
    RT.summary_writer.add_scalar('timings/a) sps', dataset.items_count / (t_end - t0), global_step) # steps per second, bigger = better
    RT.summary_writer.add_scalar('timings/b) total', t_end - t0, global_step) # time per whole cycle, lesset = better
    RT.summary_writer.add_scalar('timings/c) rollout', timings['rollout'], global_step) # time per optimization, lesser = better
    RT.summary_writer.add_scalar('timings/d) optimization', timings['optimization'], global_step) # time per optimization, lesser = better

    if 'video' in timings:
        RT.summary_writer.add_scalar('timings/e) video', timings['video'], global_step) # time per video capture
    
    RT.summary_writer.add_scalar('anneal/a) learn_rate', lr, global_step)
    RT.summary_writer.add_scalar('anneal/b) entropy_coefficient', ent_coef, global_step)
    
    RT.summary_writer.add_scalar('grad/a) total', grad_norm, global_step)

    for name, value in group_grad_norms.items(): 
        RT.summary_writer.add_scalar(f'grad/b) {name}', value, global_step)

    RT.summary_writer.add_scalar('losses/a) loss', loss, global_step)
    RT.summary_writer.add_scalar('losses/b) policy_loss', pg_loss, global_step)
    RT.summary_writer.add_scalar('losses/c) value_loss', v_loss, global_step)
    RT.summary_writer.add_scalar('losses/d) entropy_loss', entropy_loss, global_step)

    if is_action_plan_consistency_loss:
        RT.summary_writer.add_scalar('losses/e) action_plan_consistency_loss', action_plan_consistency_loss, global_step)

    if is_prediction_loss:
        RT.summary_writer.add_scalar('losses/f) prediction_loss', prediction_loss, global_step)
        
    RT.summary_writer.add_scalar('losses/g) clip_fracs', clip_fracs_af.reset(), global_step)

    if episode_stats_afs['r'].n > 0:
        assert episode_stats_afs['l'].n == episode_stats_afs['r'].n
        assert episode_stats_afs['l'].n == episode_stats_afs['lp'].n
        l = episode_stats_afs['l'].reset()
        r = episode_stats_afs['r'].reset()
        lp = episode_stats_afs['lp'].reset()
        RT.summary_writer.add_scalar('rollout/a) episode_l', l, global_step)
        RT.summary_writer.add_scalar('rollout/a) episode_r', r, global_step)
        RT.summary_writer.add_scalar('rollout/a) episode_lp', lp, global_step)
        metrics_suite['rollout/episode_l'].append(l)
        metrics_suite['rollout/episode_r'].append(r)
        metrics_suite['rollout/episode_lp'].append(lp)

    if life_stats_afs['r'].n > 0:
        assert life_stats_afs['l'].n == life_stats_afs['r'].n
        assert life_stats_afs['l'].n == life_stats_afs['lp'].n
        l = life_stats_afs['l'].reset()
        r = life_stats_afs['r'].reset()
        lp = life_stats_afs['lp'].reset()
        RT.summary_writer.add_scalar('rollout/b) life_l', l, global_step)
        RT.summary_writer.add_scalar('rollout/b) life_r', r, global_step)
        RT.summary_writer.add_scalar('rollout/b) life_lp', lp, global_step)
        metrics_suite['rollout/life_l'].append(l)
        metrics_suite['rollout/life_r'].append(r)
        metrics_suite['rollout/life_lp'].append(lp)

    RT.summary_writer.add_scalar('rollout/c) value_mean', dataset.values.mean(), global_step)
    RT.summary_writer.add_scalar('rollout/d) advantage_mean', dataset.advantages.mean(), global_step)
    RT.summary_writer.flush()

    progress = (global_step + dataset.items_count) / ceiled_global_steps_count
    LOG(f'Progress {100 * progress:.2f}% ({global_step + dataset.items_count:_} / {ceiled_global_steps_count:_})', when=not CONFIG.is_interactive)

  0%|          | 0/3 [00:00<?, ?it/s]

## Save

In [ ]:
lc = HP.launch_component()

if lc.version != 0:
    if not RT.should_save_model:
        LOG(f'Model saving skipped')
    else:
        with io.BytesIO() as b:
            # Force both compiled and non-compiled models to be saved with the same keys
            state_dict = lu.when(hasattr(RT.vision_head, '_orig_mod'), lambda: RT.vision_head._orig_mod.state_dict(), lambda: RT.vision_head.state_dict())
            torch.save(state_dict, b)
            RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='vision_head', replace=True)
            
        with io.BytesIO() as b:
            # Force both compiled and non-compiled models to be saved with the same keys
            state_dict = lu.when(hasattr(RT.encoder, '_orig_mod'), lambda: RT.encoder._orig_mod.state_dict(), lambda: RT.encoder.state_dict())
            torch.save(state_dict, b)
            RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='encoder', replace=True)
            
        with io.BytesIO() as b:
            # Force both compiled and non-compiled models to be saved with the same keys
            state_dict = lu.when(hasattr(RT.agent, '_orig_mod'), lambda: RT.agent._orig_mod.state_dict(), lambda: RT.agent.state_dict())
            torch.save(state_dict, b)
            RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)
        
        with io.StringIO() as b:
            json.dump(dataclasses.asdict(RT.agent.params), b)
            RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)
        
        with io.StringIO() as b:
            json.dump(metrics_suite, b)
            RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='metrics_suite', replace=True)
        
        with open(CONFIG.metrics_suite_fname, 'w') as f:
            json.dump(metrics_suite, f)
            LOG(f'Metrics suite saved to "{CONFIG.metrics_suite_fname}"')

    if LOG.log_fname is not None and os.path.exists(LOG.log_fname):
        import gzip, shutil
        gzip_fname = LOG.log_fname + '.gz'

        with open(LOG.log_fname, 'rb') as log_file:
            with gzip.open(gzip_fname, 'wb') as gzip_file:
                shutil.copyfileobj(log_file, gzip_file)
        
        RT.summary_writer.add_file(gzip_fname, os.path.basename(gzip_fname))
        RT.summary_writer.flush()
        LOG(f'Log file "{LOG.log_fname}" captured')

# LaunchIt!

## LAUNCH_NOTEBOOK

In [12]:
# @launchit.disable
launchit_t0 = time.monotonic()

In [13]:
# @launchit.disable
launchit_interval = time.monotonic() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    RT.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL='train',
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=['initrd', 'temp_config'], 
        disable_inds=[]
    )
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=2
Creating /home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07-launch2.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07-launch2.ipynb"


## DOCKER_LAUNCH_NOTEBOOK

In [59]:
# @launchit.disable
launchit_t0 = time.monotonic()

In [60]:
# @launchit.collect_manual_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import launch_dispatcher
    image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
    launch_request = dict(
        launch_image=image_tag,
        keep_container=False,
    )
    launch_dispatcher.LaunchRequest.run(launch_request)
# @launchit.stop

In [62]:
# @launchit.disable
launchit_interval = time.monotonic() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    RT.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL='train',
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=['initrd', 'build_docker_launch', 'manual_run_docker_launch', 'temp_config'], 
        disable_inds=[]
    )
    LOG(f'Created docker launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=3
Creating /home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07-launch3.ipynb
Created docker launch notebook "/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07-launch3.ipynb"


# System

## Create optuna trial

In [ ]:
# @launchit.collect_optuna
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK and os.path.exists('${OPTUNA_STUDY_FNAME}'):
    import re
    import launchit
    optuna_study_fname = '${OPTUNA_STUDY_FNAME}'
    optuna_study_storage = JournalStorage(JournalFileBackend(optuna_study_fname))
    optuna_study_name = '${OPTUNA_STUDY_NAME}'
    assert optuna_study_name
    assert optuna_study_name != '$' + '{OPTUNA_STUDY_NAME}', optuna_study_name
    
    optuna_study_nb_fname = re.sub('.optuna$', '.ipynb', optuna_study_fname)
    source_code = launchit.extract_source_code(optuna_study_nb_fname)
    Logging.get()(f'Extracted source code for set_hyperparamters from "{optuna_study_nb_fname}"')
    module = lu.make_module('optuna_study_hyperparameters', source_code)
    grid_search_space = lu.when(hasattr(module, 'GRID_SEARCH_SPACE'), lambda: module.GRID_SEARCH_SPACE, None)
    
    optuna_study = optuna.load_study(
        study_name=optuna_study_name, 
        storage=optuna_study_storage,
        sampler=lu.when(grid_search_space, lambda: optuna.samplers.GridSampler(grid_search_space), None)
    )
    
    optuna_trial = optuna_study.ask()
    optuna_trial.set_user_attr('MODEL_VERSION', '${MODEL_VERSION}')
    optuna_study_serial = optuna_study.user_attrs['STUDY_SERIAL']
    
    HP = module.set_hyperparameters(Hyperparameters(), optuna_study, optuna_trial)
    Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')

    assert os.path.exists(CONFIG.initrd_path)

    with open(os.path.join(CONFIG.initrd_path, 'optuna_trial.json'), 'w') as f:
        optuna_trial_dict = dict(
            trial_number=optuna_trial.number,
            study_serial=optuna_study_serial,
            study_name=optuna_study_name,
        )
        json.dump(optuna_trial_dict, f)

    with open(os.path.join(CONFIG.initrd_path, 'hyperparameters.json'), 'w') as f:
        json.dump(HP._asdict(), f)
        
elif CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK:
    hyperparameters_fname = os.path.join(CONFIG.initrd_path, 'hyperparameters.json')
    
    with open(hyperparameters_fname, 'r') as f:
        HP = Hyperparameters.from_dict(json.load(f))
        Logging.get()(f'HP loaded from "{hyperparameters_fname}"')
        Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')


## Run optuna trial

In [ ]:
# @launchit.collect_optuna_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    # optuna_study and optuna_trial are created in cell above
    assert optuna_study is not None
    assert optuna_trial is not None

    import launch_dispatcher
    short_image_tag = '${MODEL_NAME}' + ':' + '${MODEL_VERSION}'
    image_tag = os.path.join(CONFIG.docker_registry, short_image_tag)
    launch_request = dict(
        launch_image=image_tag,
        result_fname=os.path.join(project_root_path, CONFIG.relative_metrics_suite_fname),
        keep_container=False,
    )
    launch_result_metadata, launch_result_body = launch_dispatcher.LaunchRequest.run(launch_request)
    
    with open(CONFIG.self_fname + '.out', mode='wt') as out_file:
        if launch_result_metadata['is_ok']:
            if launch_result_body:
                decisive_metric = '${OPTUNA_DECISIVE_METRIC}'
                
                with io.BytesIO(launch_result_body) as b:
                    launch_result_dict = json.load(b)
    
                if launch_result_dict.get(decisive_metric, []):
                    decisive_metric_arr = np.array(launch_result_dict[decisive_metric])
                    scalar_result = lu.when(len(decisive_metric_arr) > 0, lambda: decisive_metric_arr[-1], 0) # reduce to just scalar e.g. via mean, sum, last
                    optuna_study.tell(optuna_trial, scalar_result, state=optuna.trial.TrialState.COMPLETE)
    
                    message = f'Trial {optuna_trial.number} ({short_image_tag}) finished with value: {scalar_result} and parameters: {optuna_trial.params}'
                    
                    try:
                        best_trial = optuna_study._get_best_trial(deepcopy=False)
                        message += f'. Best is trial {best_trial.number} with value: {best_trial.value}'
                    except ValueError:
                        # If no feasible trials are completed yet, study.best_trial raises ValueError
                        pass

                    out_file.write(message)
                else:
                    optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                    out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty/missing "{decisive_metric}"')
            else:
                optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty result body: {launch_result_metadata=}')
        else:
            optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
            out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) failed: {launch_result_metadata=}')

    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, message="To exit: use 'exit'")
    sys.exit(0)

## Save artifact registry cache

In [ ]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig':
    if CONFIG.artifact_registry_cache:
        with open(CONFIG.artifact_registry_cache_fname, 'wb') as f:
            pickle.dump(CONFIG.artifact_registry_cache, f)

        Logging.get()(f'Artifact registry cache saved to "{CONFIG.artifact_registry_cache_fname}"')
    else:
        Logging.get()(f'Artifact registry cache is empty, nothing to save')
# @launchit.stop

## Docker build

In [ ]:
# @launchit.collect_build_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import subprocess
    import tempfile

    assert os.path.exists(os.path.join(build_project_root_path, CONFIG.relative_self_fname))

    dockerfile_content = f'''
FROM {os.path.join(CONFIG.docker_registry, 'neurolab_source:latest')}
WORKDIR /neurolab
RUN --mount=type=secret,id=neurolab_deploy_key,target=/tmp/neurolab_deploy_key,uid=1000 \\ 
    export GIT_SSH_COMMAND="ssh -i /tmp/neurolab_deploy_key -o IdentitiesOnly=yes" && \\
    git pull --rebase
WORKDIR /neurolab/{CONFIG.subproject_name}
COPY --chown=1000:1000 {CONFIG.relative_self_fname} .
'''
    if os.path.exists(CONFIG.initrd_path):
        dockerfile_content += f'''
RUN mkdir -p /neurolab/{CONFIG.relative_initrd_path}
COPY --chown=1000:1000  {CONFIG.relative_initrd_path} /neurolab/{CONFIG.relative_initrd_path}
'''

    dockerfile_content += f'''
CMD ["papermill", "{os.path.basename(CONFIG.self_fname)}", "{os.path.basename(CONFIG.self_fname)}"]
    '''
    with tempfile.NamedTemporaryFile(mode='wt', suffix='.Dockerfile') as dockerfile:
        with open(dockerfile.name, 'w') as f:
            f.write(dockerfile_content)

        image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
        subprocess.run(
            [
                'docker', 'buildx', 'build', '-f', dockerfile.name, '-t', image_tag, build_project_root_path, 
                '--load', '--no-cache', '--network=host', 
                # --add-host is used to overcome problems with VPN (when VPN enabled build process cannot resolve nexus host)
                '--add-host', 'nexus=127.0.0.1',
                # use SSH to connect to GIT to overcome Anti-DDOS barriers
                '--secret', 'id=neurolab_deploy_key,src=/home/misha/.ssh/neurolab_deploy_key',
            ],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
        subprocess.run(
            ['docker', 'push', image_tag],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
# @launchit.stop